**Resumen:** El presente proyecto pretende analizar y predecir el precio de los alquileres de departamentos en la Ciudad Autónoma de Buenos Aires (CABA) utilizando un conjunto de datos reales obtendidos a partir de la web del gobierno de la ciudad. A través del análisis exploratorio de datos (EDA), se identificarán las variables más relevantes para los alquileres, como ubicación, superficie y cantidad de ambientes. Posteriormente, se desarrollará un modelo de predicción que permita estimar el precio de alquiler basándose en dichas características. Los hallazgos clave buscarán ofrecer insights útiles para propietarios, inquilinos y empresas del sector inmobiliario.

**Objetivo:** Determinar cuáles son las variables más influyentes en el precio de alquiler de departamentos en CABA y desarrollar un modelo predictivo que permita estimar el costo de alquiler a partir de estas características.

**Cotexto empresarial:** La empresa InmoPredict poseedora de una página web en la que se publican alquileres de la Ciudad Autónoma de Buenos Aires, busca expandir su oferta de herramientas analíticas para el mercado inmobiliario. Con este trabajo, se pretende desarrollar un modelo predictivo que ayude a propietarios a establecer precios competitivos y justos, así como a los inquilinos a identificar las mejores opciones según su presupuesto y requerimientos. Este enfoque permitirá posicionar a InmoPredict como un referente en la toma de decisiones basada en datos en el sector.

**Hipótesis principal:** El precio de alquiler en dólares para departamentos con características similares se mantiene relativamente estable en el tiempo, ajustando por la inflación en dólares.

**Hipótesis secundarias:**
*   El precio de alquiler en dólares aumenta proporcionalmente con la superficie cubierta y la cantidad de ambientes.
*   Hay mayor oferta de inmuebles de pocos ambientes y superficies relativamente chicas.

**Información del DataSet:** Los datos fueron obtenidos a través de la página oficial del gobierno de la ciudad de Buenos Aires. A continuación se indicará qué representa cada vaiable del mismo.
- id: identificación única.
- antig: antiguedad del inmueble.
- barrio: barrio donde se encuentra el inmueble.
- dirección: dirección donde se encuentra el inmueble.
- lat: latitud de la ubicación del inmueble.
- lng: longitud de la ubicación del inmueble.
- fecha_pub_aviso: día, mes y año en qué se publicó el alquiler.
- exp: valor de las expensas del inmueble.
- sup_total: superficie total del inmueble.
- sup_cub: superficie cubierta del inmueble.
- alq_usd: valor del alquiler en dólares.
- alq_arg: valor del alquiler en pesos argentinos.
- alq_dolar: valor del alquiler en dólares.
- alq_peso: valor del alquiler en pesos argentinos.
- ascen: presencia o ausencia de ascensor en el edificio.
- gimnasio: presencia o ausencia de gimnasio en el inmueble o edificio.
- lavadero: presencia o ausencia de lavadero en el inmueble.
- sum: presencia o ausencia de salón de usos múltiples en el inmueble o edificio.
- pileta: presencia o ausencia de pileta en el inmueble o edificio.
- balc: presencia o ausencia de balcón en el inmueble.
- parrilla: presencia o ausencia de parrilla en el inmueble o edificio.
- aire: presencia o ausencia de aire acondicionado en el inmueble.
- amoblado: presencia o ausencia de mobiliario en el inmueble.
- solarium: presencia o ausencia de solarium e la zona de la pileta.
- vigilancia: presencia o ausencia de personal de seguridad y vigilancia en el edificio.
- cochera: presencia o ausencia de cochera en el inmueble o edificio.
- frente: característica presente en inmuebles con vista a la calle.
- contrafrente: característica presente en inmuebles ubicados en la parte de atrás del edificio o vista al patio del mismo.
- amb: cantidad de ambientes del inmueble.
- mes: mes y año de publicación del alquiler.

# **1. IMPORTACIÓN DE LIBRERÍAS**

In [ ]:
from sklearn.experimental import enable_halving_search_cv  # noqa  Import enable_halving_search_cv before importing HalvingGridSearchCV

#Librerías pricipales (manipulación de datos y gráficos)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

# Librarías para machine learning
from sklearn.model_selection import train_test_split, GridSearchCV, HalvingGridSearchCV, KFold, cross_val_score # Now you can import HalvingGridSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, explained_variance_score
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_regression
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
import xgboost as xgb

# Librarías para correlaciones
from scipy.stats import spearmanr, kendalltau, stats

# Librarías para hiperparámetros
from hyperopt import fmin, tpe, hp

# Librarías para escalar variables
from sklearn.preprocessing import MinMaxScaler

# **2. CARGA DE DATOS**

In [ ]:
#Clonar el repositorio
!git clone https://github.com/MVernetti/Alquileres_CABA.git

In [ ]:
#Cambiar al directorio de mi repositorio
%cd T/content/TPFinal_alquileresCABA/Archivos CSV/Alquileres_CABA

#Listar los archivos del repositorio
!ls

In [ ]:
#Leer archivo CSV.
data=pd.read_csv("/content/Alquileres_CABA/2alquileres_CABA.csv", encoding='latin-1', sep='|')


# **3. EXPLORACIÓN DE LOS DATOS**

In [ ]:
#Crear dataframe
df=pd.DataFrame(data)

In [ ]:
#Se muestra encabezado
print(df.head())

In [ ]:
# Se muestran los nombres de las columnas
print(df.columns)

In [ ]:
#Se muestran cantidad de filas y columnas
print(df.shape)

In [ ]:
#Se muestra información general
print(df.info())

Se puede observa que todas las variables, excepto "id", presentan valores nulos. También es importante destacar que la variable cochera tiene una alta proporción de valores nulos.

In [ ]:
#Se muestran los principales estadísticos
df.describe()



## **3.1** *Variables numéricas*

### **3.1.1 Variable Alq_dolar**

In [ ]:
#Boxplot de la variable alq_dolar
sns.boxplot(x='alq_dolar', data=df)
plt.title('Distribución del precio en dólares')
plt.xlabel('Precio en dólares')
plt.show()

En el diagrama de cajas se observan muchos outliers.
A continuación se realizará un histograma limitando sus valores para observar cómo se distribuyen los valores en el rango de mayor concentración de datos.

In [ ]:
#Histograma
plt.hist(df['alq_dolar'], bins=20,range=(0, 2000))
plt.xlabel('alq_dolar')
plt.ylabel('Cantidad de departamentos')
plt.title('Distribución del precio en dólares')

Se observa una distribución sesgada positiva.

In [ ]:
#Diagrama de disperción entre alq_dolar y alq_peso
plt.scatter(df['alq_peso'], df['alq_dolar'])
plt.xlabel('alq_peso')
plt.ylabel('alq_dolar')
plt.title('Diagrama de dispersión entre alq_dolar y alq_peso')

En el gráfico se observa una correlacion lineal positiva clara entre las variables que ya se anticipó en la matriz de correlación.

### **3.1.2 Variable Alq_peso**

In [ ]:
#Boxplot de la variable alq_dolar
sns.boxplot(x='alq_peso', data=df)
plt.title('Distribución del precio en pesos')
plt.xlabel('Precio en pesos')
plt.show()

En el diagrama de cajas se observan muchos outliers.
A continuación se realizará un histograma limitando sus valores para observar cómo se distribuyen los valores en el rango de mayor concentración de datos.

In [ ]:
#Histograma
plt.hist(df['alq_peso'], bins=20,range=(0, 100000))
plt.xlabel('alq_peso')
plt.ylabel('Cantidad de departamentos')
plt.title('Distribución del precio en pesos')

Se observa una distribución sesgada positiva.
Posiblemente exista alguna relación entre el alquiler en pesos y en dólares, por lo que se hará un diagrama de disperción entre ambas variables para observar la presencia de potenciales patrones.

### **3.1.3 Variable 'antiguedad'**

In [ ]:
# Crear el histograma
plt.hist(df['antig'], bins=50, edgecolor='black')

# Personalizar el gráfico
plt.title('Distribución de la Antigüedad')
plt.xlabel('Antigüedad (años)')
plt.ylabel('Cantidad de inmuebles')

# Mostrar el gráfico
plt.show()

In [ ]:
#Boxplot
sns.boxplot(x='antig', data=df)
plt.title('Distribución de la Antigüedad de los inmuebles')
plt.show()

En los gráficos se puede observar que la mayor cantidad de datos se concentran en el intervalo [0,100]. También se destacanoutliers y valores erroneos puesto que un inmueble no puede tener una antiguedad de 2000 años.

La antiguedad puede estar relacionada con el barrio en el que se encuentra el inmueble, por lo que se graficarán estas dos variables para observar su relación.

In [ ]:
# Filtrar los datos para que la antigüedad sea menor o igual a 100
df_filtrado = df[df['antig'] <= 100]

# Agrupar los datos por barrio
grouped = df_filtrado.groupby('barrio')

# Iterar sobre cada grupo y crear un histograma
for barrio, group in grouped:
    plt.figure(figsize=(6, 4))
    sns.histplot(data=group, x='antig', bins=20, kde=True)  # Ajusta el número de bins según tus datos
    plt.title(f'Histograma de Antigüedad - Barrio: {barrio} (n={len(group)})')
    plt.xlabel('Antigüedad (años)')
    plt.ylabel('Frecuencia')
    plt.show()

En los gráficos se puede observar que la distribución de la antiguedad es muy diferente en cada barrio. No obstante, pareciera que ninguno tiene distribución normal.

### **3.1.4 Variable 'superficie'**

In [ ]:
#Boxplot de la variable superficie
sns.boxplot(x='sup_cub', data=df)
plt.title('Distribución de la superficie en m2')
plt.xlabel('Superficie (m2)')
plt.show()

En el boxplot se observan muchos outliers y datos erroneos.

In [ ]:
#Histograma de la variable superficie
plt.hist(df['sup_cub'], bins=20,range=(0, 500))
plt.xlabel('superficie')
plt.ylabel('Cantidad de departamentos')
plt.title('Distribución de la superficie de los departamentos')
plt.show()

En el histograma con límite superior de 500 metros cuadrados, se observa que la mayor concentración de datos se encuentra en el intervalo [0,100]. La distribución es sesgada derecha y no pareciera ser normal.

La superficie de los inmuebles podría estar relacionada con la cantidad de ambientes, por lo que se graficarán ambas variables en un diagrama de dispersión.

In [ ]:
# Filtrar los datos para superficies menores a 100 y ambientes menores a 10
df_filtrado = df[(df['sup_cub'] < 100) & (df['amb'] < 10)]

# Crear el diagrama de dispersión
plt.scatter(df_filtrado['sup_cub'], df_filtrado['amb'])

# Personalizar el gráfico
plt.title('Diagrama de Dispersión: Superficie Cubierta vs. Cantidad de Ambientes')
plt.xlabel('Superficie Cubierta (m²)')
plt.ylabel('Cantidad de Ambientes')

plt.show()

No pareciera observarse una relación entre las variables, aunque podría mejorar con la limpieza de datos.

Otra relación importante de  la variable superfie podría ser con el precio de alquiler.

In [ ]:
# Filtrar los datos para superficies menores a 100 y alquiler en dolares menores a 2000
df_filtrado = df[(df['sup_cub'] < 100) & (df['alq_dolar'] < 2000)]

# Crear el diagrama de dispersión
plt.scatter(df_filtrado['sup_cub'], df_filtrado['alq_dolar'])

# Personalizar el gráfico
plt.title('Diagrama de Dispersión: Superficie Cubierta vs. Alquiler en dólares')
plt.xlabel('Superficie Cubierta (m²)')
plt.ylabel('Alquiler en dólares')

plt.show()

Aquí se puede observar una correlación positiva aunque no parece estrictamente lineal.

### **3.1.5 Variable 'ambientes'**

In [ ]:
#Gráfico de barras de la variable ambientes
plt.figure(figsize=(20, 6))
sns.countplot(x='amb', data=df)
plt.title('Distribución de la cantidad de ambientes')
plt.xlabel('Cantidad de ambientes')
plt.ylabel('Cantidad de propiedades')
plt.show()



En el gráfico de barras se puede observar que la mayor concentración de datos está entre 0 y 6 ambientes. También se observan muchos datos erroneos. La distribución es sesgada positiva.

## **3.2** *Variables categóricas*

### **3.2.1 Variable 'barrio'**

In [ ]:
#Conteo por barrio
conteo_por_barrio = df.groupby('barrio').size()
pd.set_option('display.max_rows', None)
print(conteo_por_barrio)

In [ ]:
# Contar la frecuencia de cada barrio
conteo_barrios = df['barrio'].value_counts()

# Gráfico de barras con Matplotlib
plt.figure(figsize=(14, 6))
conteo_barrios.plot(kind='bar')
plt.title('Distribución de Barrios')
plt.xlabel('Barrio')
plt.ylabel('Cantidad')
plt.show()

Se puede observar que hay mucha diferencia en la cantidad de datos por barrio. Si bien a priori se podría decir que algunos barrios tienen mayor oferta de inmuebles, hay que considerar que si la cantidad de datos en algunos es muy pequeña, es probable que no sean datos representativos.

### **3.2.2 Variable 'gimnasio'**

In [ ]:
# Gráfico de barras de la variable gimnasio
sns.countplot(x='gimnasio', data=df)

# Personalizando el gráfico
plt.title('Distribución de la variable gimnasio')
plt.xlabel('Tiene gimnasio (0: No, 1: Sí)')
plt.ylabel('Cantidad de inmuebles')

plt.show()

Se puede obsevar que la proporción de inmuebles sin gimnasio es bastante mas grande a la que tiene gimnasio.


### **3.2.3 Variable 'sum'**

In [ ]:
# Gráfico de barras de la variable sum
sns.countplot(x='sum_', data=df)

# Personalizando el gráfico
plt.title('Distribución de la variable sum')
plt.xlabel('Tiene sum (0: No, 1: Si)')
plt.ylabel('Cantidad de inmuebles')

plt.show()

Se puede observar que hay mas inmuebles sin sum que los que tienen sum.

### **3.2.4 Variable 'pileta'**

In [ ]:
# Gráfico de barras de la variable pileta
sns.countplot(x='pileta', data=df)

# Personalizando el gráfico
plt.title('Distribución de la variable pileta')
plt.xlabel('Tiene pileta (0: No, 1: Si)')
plt.ylabel('Cantidad de inmuebles')

plt.show()

Se observa que hay una mayor proporción de inmuebles sin pileta que los que tienen pileta.

### **3.2.5 Variable 'balcon'**

In [ ]:
# Gráfico de barras de la variable balc
sns.countplot(x='balc', data=df)

# Personalizando el gráfico
plt.title('Distribución de la variable balcón')
plt.xlabel('Tiene balcón (0: No, 1: Si)')
plt.ylabel('Cantidad de inmuebles')

plt.show()

Se observa que hay una mayor proporción de inmuebles con balcón que los que no tienen. A grandes rasgos aproximadamente dos terceras partes de los inmuebles tienen balcón.

### **3.2.6 Variable 'parrilla'**

In [ ]:
# Gráfico de barras de la variable parrilla
sns.countplot(x='parrilla', data=df)

# Personalizando el gráfico
plt.title('Distribución de la variable parrilla')
plt.xlabel('Tiene parrilla (0: No, 1: Si)')
plt.ylabel('Cantidad de inmuebles')

plt.show()

Aquí se puede observar que la mayoría de los departamentos no tienen parrilla.

### **3.2.7 Variable 'aire'**

In [ ]:
# Gráfico de barras de la variable aire
sns.countplot(x='aire', data=df)

# Personalizando el gráfico
plt.title('Distribución de la variable parrilla')
plt.xlabel('Tiene parrilla (0: No, 1: Si)')
plt.ylabel('Cantidad de inmuebles')

plt.show()

se observa que hay mas inmuebles sin aire, no obstante, la diferencia no es grande.

### **3.2.8 Variable 'amoblado'**

In [ ]:
# Gráfico de barras de la variable amoblado
sns.countplot(x='amoblado', data=df)

# Personalizando el gráfico
plt.title('Distribución de la variable amoblado')
plt.xlabel('Tiene amoblado (0: No, 1: Si)')
plt.ylabel('Cantidad de inmuebles')

plt.show()

Se observa que la mayoría de los inmuebles no son amoblados. La diferencia entre cantidad de amoblados y no amoblados es grande.

### **3.2.9 Variable 'vigilancia'**

In [ ]:
# Gráfico de barras de la variable amoblado
sns.countplot(x='vigilancia', data=df)

# Personalizando el gráfico
plt.title('Distribución de la variable vigilancia')
plt.xlabel('Tiene vigilancia (0: No, 1: Si)')
plt.ylabel('Cantidad de inmuebles')

plt.show()

Se observa que la mayoría de los departamentos no tienen vigilancia.

# **4 PREPROCESAMIENTO DE DATOS**

In [ ]:
#Se renombran columnas
df = df.rename(columns={'antig':'antiguedad','sup_cub':'superficie','ascen': 'ascensor', 'sum_': 'sum','balc':'balcon','amb':'ambientes','mes':'fecha_publ'})

In [ ]:
#Cambiar formato de visualización de la columna 'mes'
df['fecha_publ'] =pd.to_datetime(df['fecha_publ'].str.replace('m','-'),format='%Y-%m')


La columna 'cochera' tiene el 35% de valores nulos, como es un porcentaje alto se eliminará dicha variable.

In [ ]:
#Se elimina la columna cochera
df=df.drop(columns=['cochera'])


Hay dos pares de columnas que muestran los mismos datos: 'alq_peso' y 'alq_arg' es uno de ellos y el otro es 'alq_dolar' y 'alq_usd'. Para eliminar columnas repetidas se van a contar los valores nulos de cada una, se comparará si las celdas de la columna con mas valores nulos tambien son valores nulos en la columna con menos nulos (esto para no eliminar una columna con valores que puedan usarse en la columna que quedará en el dataframe) y por útimo eliminaremos las columnas.

In [ ]:
#Conteo de nulos de dos columnas iguales
nulos_alq_peso = df['alq_peso'].isnull().sum()
nulos_alq_arg = df['alq_arg'].isnull().sum()
print('Cantidad de valores nulos de la columna alq_peso: ',nulos_alq_peso)
print('Cantidad de valores nulos de la columna alq_arg: ',nulos_alq_arg)

nulos_alq_dolar=df['alq_dolar'].isnull().sum()
nulos_alq_usd=df['alq_usd'].isnull().sum()
print('Cantidad de valores nulos de la columna alq_dolar: ',nulos_alq_dolar)
print('Cantidad de valores nulos de la columna alq_usd: ', nulos_alq_usd)



In [ ]:
# Se verifica si los valores nulos en la columna con menos nulos también son nulos en la columna con más nulos
mismos_nulos_pesos = df['alq_peso'].isnull().isin(df['alq_arg'].isnull()).all()
print(mismos_nulos_pesos)

mismos_nulos_dolares = df['alq_dolar'].isnull().isin(df['alq_usd'].isnull()).all()
print(mismos_nulos_dolares)

In [ ]:
#Se eliminan columnas con mas valores nulos
df=df.drop(columns=['alq_arg','alq_usd'])


In [ ]:
#Contar filas duplicadas
cantidad_duplicados=df.duplicated().sum()
print(cantidad_duplicados)

Se decide eliminar las filas duplicadas puesto que provocarían distorción en la realidad de los alquileres.

In [ ]:
#Eliminar filas duplicadas
df=df.drop_duplicates()

Se verifica si hay valores de id duplicados porque se podría tratar de los mismos inmuebles.

In [ ]:
#Verificar que la columna id no tenga valores duplicados
cantidad_duplicados_id=df['id'].duplicated().sum()
print(cantidad_duplicados_id)

Para tratar las filas con la misma id, se agrupará por duplicados de id y dirección, porque se trataría del mismo inmueble. Luego se verá si hay datos que se complementan y por último se eliminarán las filas con mas valores nulos.

In [ ]:
#Ordenar por id ascendente
df=df.sort_values('id')

In [ ]:
#Filtrar las filas de id duplicadas
df_duplicados_id=df[df['id'].duplicated(keep=False)]
df_duplicados_id=df_duplicados_id.sort_values('id')


In [ ]:
# Identifica las filas donde se repiten las columnas id y dirección
duplicados_id_direccion = df.duplicated(subset=['id', 'direccion'], keep=False)

# Filtra el DataFrame para obtener solo las filas duplicadas
filas_duplicadas_id_dirección = len(df[duplicados_id_direccion])
print(filas_duplicadas_id_dirección)

In [ ]:
#Agrupar por id y dirección
agrupacion_por_id_direccion=df.groupby(['id','direccion'])

In [ ]:
#Se define la función para eliminar las filas filas duplicadas donde los valores nulos de la fila con menos nulostambién están presentes como nulos en las otras filas duplicadas.

def eliminar_duplicados_mismos_nulos(df, columnas=['id', 'direccion']):


    # Ordenar por cantidad de valores nulos por fila de forma ascendente
    df['nulos'] = df.isnull().sum(axis=1)
    df = df.sort_values('nulos')

    # Marcar duplicados considerando solo las columnas especificadas
    df['duplicado'] = df.duplicated(subset=columnas)

    # Crear una nueva columna para indicar si los valores nulos de la fila a eliminar son exactamente iguales en la fila que se queda.
    df['mismos_nulos'] = False
    for index, row in df.iterrows():
        if row['duplicado']:
            # Obtener la fila anterior (que tiene menos nulos)
            fila_anterior = df.iloc[index-1]
            # Verificar si los valores nulos de la fila actual son exactamente iguales a los de la fila anterior.
            df.loc[index, 'mismos_nulos'] = (row.isnull() == fila_anterior.isnull()).all()

    # Eliminar las filas duplicadas donde los patrones de nulos son iguales
    df = df[~((df['duplicado']) & (df['mismos_nulos']))]

    # Eliminar las columnas auxiliares
    df = df.drop(['nulos', 'duplicado', 'mismos_nulos'], axis=1)

    return df

In [ ]:
#Elimina filas duplicadas basado en las columnas 'id' y 'direccion',solo si no hay valores nulos en ninguna columna de esa fila.

def eliminar_duplicados_sin_nulos_en_ninguna_columna(df):
  # Eliminar duplicados directamente, considerando solo 'id' y 'direccion'
  df_sin_duplicados = df.drop_duplicates(subset=['id', 'direccion'])

  return df_sin_duplicados

# Aplicando la función al DataFrame original
df = eliminar_duplicados_sin_nulos_en_ninguna_columna(df)

Ahora se verifica si aún hay valores de id repetidos.

In [ ]:
#Verificar que la columna id no tenga valores duplicados
cantidad_duplicados_id=df['id'].duplicated().sum()
print(cantidad_duplicados_id)

Si bien hay valores de id repetidos, no se puede afirmar que se trate de los mismos inmuebles por lo que no se eliminarán.

In [ ]:
#Crear id única
df['id_nueva'] = df.index

In [ ]:
#Se eliminan columnas que no se necesitan
df=df.drop(columns=['direccion','lat','lng','fecha_pub_aviso','exp','sup_tot','solarium','frente','contrafrente','id', 'ascensor'])


In [ ]:
#Renombrar columna id_nueva por id
df=df.rename(columns={'id_nueva':'id'})

Eliminar las filas en las que alq_dolar y alq_peso sean valores nulos y ademas o superficie es nulo o ambientes es nulo, puesto que faltaría información muy importante.

In [ ]:
# Obtener un índice booleano de las filas a eliminar
filas_a_eliminar = (df['alq_dolar'].isnull()) & (df['alq_peso'].isnull()) & \
                   ((df['superficie'].isnull()) | (df['ambientes'].isnull()))

# Eliminar las filas utilizando el método drop
df = df.drop(df.index[filas_a_eliminar])


A continuación se cambia el orden de las columnas

In [ ]:
# Cambio de orden de las columnas
df = df.reindex(columns=['id', 'alq_dolar','alq_peso', 'superficie','antiguedad','ambientes','barrio','gimnasio',
                         'lavadero','sum','pileta','balcon','parrilla','aire','amoblado','vigilancia', 'fecha_publ'])

In [ ]:
#Matriz de correlación de Pearson
eliminar_columnas=['id','barrio','fecha_publ']
df_filtrado=df.drop(columns=eliminar_columnas,axis=1)
corr_matrix = df_filtrado.corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f')
plt.title('Matriz de Correlación')
plt.show()

En la Matriz de correalción de Pearson se observan relaciones lineales muy bajas con la variable principal (alq_dolar), se espera que la correlación entre variables aumente luego de la limpieza de datos.
La relación entre las variables alq_dolar y alq_peso -como era de esperar- es muy fuerte.

## **4.1** *Variables categóricas*

En primer lugar se realizará el preprocesamiento de las variables categóricas.

### **4.1.1 Variable 'barrio'**



In [ ]:
#Conteo de valores nulos de barrio
nulos_barrio=df['barrio'].isnull().sum()
print(nulos_barrio)

In [ ]:

# Cambia nombre de barrio erroneo
df.loc[df['barrio'] == '-58.48424710000001', 'barrio'] = 'otro'


In [ ]:
# Reemplazar valores nulos en la columna "barrio" por "otro"
df['barrio'].fillna('otro', inplace=True)

In [ ]:
# Agrupar por barrio
agrupacion_por_barrio = df.groupby('barrio')
print(agrupacion_por_barrio)

Algunos de los barrios se encuentran separados pero podrìan considerarse un mismo barrio, por lo que se les cambiará el nombre.

In [ ]:
#Renombrar barrios
df['barrio'] = df['barrio'].replace({'palermo chico': 'palermo', 'palermo hollywood': 'palermo', 'palermo soho': 'palermo', 'palermo viejo': 'palermo', 'villa devoto': 'villa crespo y villa devoto', 'villa crespo': 'villa crespo y villa devoto','chacarita':'chacarita y villa ortuzar','villa ortuzar':'chacarita y villa ortuzar','flores':'zona sur','floresta':'zona sur','velez sarsfield':'zona sur','barracas':'barracas y parque patricios','parque patricios':'barracas y parque patricios','san nicolas':'centro','retiro':'centro','monserrat':'centro','boca':'san telmo y la boca','san telmo':'san telmo y la boca'})

In [ ]:
#Conteo por barrio
conteo_por_barrio = df.groupby('barrio').size()
pd.set_option('display.max_rows', None)
print(conteo_por_barrio)

Se creará una nueva columna para dividir los barrios en tres categorías (0,1 y 2), donde 0 corresponde a los barrios mas caros y 2 a los barrios mas baratos. Luego se eligirá que variable funciona mejor en el modelo.

In [ ]:
# Crear columna 'categoria_barrio'

# Se define un diccionario para mapear los barrios a sus categorías (1, 2 y 3)
mapeo = {'puerto madero': 3, 'recoleta':3, 'belgrano':3, 'barrio norte':3,'las cañitas':3,'palermo':3 ,'centro':3,'catalinas':3,'nuñez': 3,
         'san telmo y la boca':2 ,'caballito': 2,'colegiales': 2,'chacarita y villa ortuzar': 2,'almagro': 2,'abasto': 2,'balvanera': 2,'saavedra': 2,'boedo': 2,'congreso': 2,'san cristobal': 2,'liniers': 2,'villa crespo y villa devoto': 2,'parque centenario': 2,'villa real': 2,'otro':2,
         'villa riachuelo' : 1, 'barracas y parque patricios' : 1, 'mataderos' : 1, 'versalles' : 1, 'pompeya' : 1, 'villa del parque' : 1, 'villa lugano' : 1, 'villa luro' : 1, 'villa mitre' : 1, 'villa pueyrredon' : 1, 'parque avellaneda' : 1, 'parque chas' : 1, 'tribunales' : 1, 'once' : 1, 'monte castro' : 1, 'zona sur' : 1, 'paternal' : 1, 'floresta' : 1, 'villa santa rita' : 1, 'villa soldati' : 1,'constitucion' : 1, 'villa urquiza' : 1, 'coghlan': 1, 'parque chacabuco' : 1, 'agronomia' : 1 }

# Se crea la nueva columna
df['categoria_barrio'] = df['barrio'].map(mapeo)

print(df.head())

### **4.1.2 Variables  'gimnasio', 'lavadero', 'sum', 'pileta', 'balcon', 'parrilla', 'aire', 'amoblado' y 'vigilancia'**

Se reemplazarán los valores nulos por 0, asumiendo que los inmuebles no poseen esta característica.

In [ ]:
# Verificar valores nulos en variables de amenidades
amenidades = ['gimnasio', 'lavadero', 'sum', 'pileta', 'balcon', 'parrilla', 'aire', 'amoblado', 'vigilancia']
print('Valores nulos por amenidad:')
print(df[amenidades].isnull().sum())

# Reemplazar nulos por 0 (el inmueble no posee la característica)
df[amenidades] = df[amenidades].fillna(0)

## **4.2** *Variables numéricas*

### **4.2.1 Variable  'antiguedad'**

Los inmuebles no pueden tener una antigüedad negativa, y leyendo sobre las construcciones de la ciudad de Buenos Aires, es muy probable que los inmuebles con antigüedad mayor a 100 años tambien sean datos erróneos.

In [ ]:
#Contar valores por fuera del rango 0-100

# Contar valores mayores a 100
cantidad_de_valores_mayor_100 = (df['antiguedad'] > 100).sum()
print('La cantidad de inmuebles con antigüedad mayor a 100 años es: ',cantidad_de_valores_mayor_100)

# Contar valores menores a 0
cantidad_de_valores_menor_0 =(df['antiguedad'] < 0).sum()
print('La cantidad de inmuebles con antigüedad menor a 0 años es: ',cantidad_de_valores_menor_0)


In [ ]:
#Reemplazar los valores mayores a 100 por valores nulos

df['antiguedad'] = np.where(df['antiguedad'] > 100, np.nan, df['antiguedad'])

In [ ]:
#Reemplazar los valores maenores a 0 por valores nulos

df['antiguedad'] = np.where(df['antiguedad'] < 0, np.nan, df['antiguedad'])

Los datos no tienen distribución normal por lo que la media no es conveniente para rellenar valores nulos.

In [ ]:
#boxplot de la variable antigüedad
sns.boxplot(x='antiguedad', data=df)
plt.title('Distribución de la Antigüedad de los inmuebles')
plt.show()

Se pueden observar en el boxplot muchos outliers que se eliminarán a continuación, ya que por ser valosres muy altos se tratarían de datos erroneos o ecxepciones.

In [ ]:
#Eliminar outliers

# Calcular los cuartiles
Q1 = df['antiguedad'].quantile(0.25)
Q3 = df['antiguedad'].quantile(0.75)

# Calcular el rango intercuartílico (IQR)
IQR = Q3 - Q1

# Definir los límites para identificar outliers
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Identificar los outliers
outliers = df[(df['antiguedad'] < lower_bound) | (df['antiguedad'] > upper_bound)]

# Crear una nueva columna para indicar si un dato es outlier
df['is_outlier'] = ((df['antiguedad'] < lower_bound) | (df['antiguedad'] > upper_bound)).astype(int)

# Reemplazar los outliers por NaN directamente en el DataFrame original
df.loc[(df['antiguedad'] < lower_bound) | (df['antiguedad'] > upper_bound), 'antiguedad'] = np.nan

# Eliminar la columna 'is_outlier'
df.drop('is_outlier', axis=1, inplace=True)



Para completar los datos faltantes de antiguedad se verá la distribución de dicha variable con respecto a los barrios ya que hay barrios mas nuevos que otros.

En los gráficos no se observa normalidad, por lo que se utilizará la mediana para imputar los valores nulos.

In [ ]:
#Reemplazar valores nulos de antiguedad por la mediana de cada barrio
df['antiguedad'] = df.groupby('barrio')['antiguedad'].transform(lambda x: x.fillna(x.median()))

### **4.2.2 Variable 'ambientes'**

In [ ]:
#Filtrar las filas de mayor cantidad de ambientes
mas_ambientes= df[(df['ambientes'] > 7)][['id','superficie','alq_dolar','alq_peso','ambientes']].sort_values('ambientes', ascending=False)
print(mas_ambientes)

Observando los datos, los valores de la variable 'ambientes' mayores a 15n serían erroneos, por lo que se reemplazarán por valores nulos.

In [ ]:
#Reemplazar los valores mayores a 15 por valores nulos
df['ambientes'] = np.where(df['ambientes'] > 15, np.nan, df['ambientes'])

Se verifica la cantidad de datos donde ambientes es igual a cero ya que serían datos erroneos. Luego se los reemplaza por valores nulos.

In [ ]:
#Contar la cantidad de datos donde ambientes es igual a 0
cantidad_ceros = (df['ambientes'] == 0).sum()
print("Cantidad de datos iguales a cero:", cantidad_ceros)


In [ ]:
#Reemplazar los 0 por valores nulos
df['ambientes'] = np.where(df['ambientes'] == 0, np.nan, df['ambientes'])

Se considera que si la cantidad de ambientes es mayor a 7 y la superficie es menor a 120, se trata de valores erroneos, por lo que se reemplazarán por valores nulos.

In [ ]:
# Aplica la condición y asigna valores nulos cuando ambientes es mayor que 7 y superficie menor a 120
df.loc[(df['ambientes'] > 7) & (df['superficie'] < 120), 'ambientes'] = np.nan

In [ ]:
#Conteo de valores nulos ambientes
nulos_ambientes=df['ambientes'].isnull().sum()
print(nulos_ambientes)

In [ ]:
#Gráfico de barras de la variable ambientes
plt.figure(figsize=(20, 6))
sns.countplot(x='ambientes', data=df)
plt.title('Distribución de la cantidad de ambientes')
plt.xlabel('Cantidad de ambientes')
plt.ylabel('Cantidad de propiedades')
plt.show()

La distribución de la variable es sesgada positiva, por lo que se utilizará la mediana para la imputación de datos.

In [ ]:
#Mediana
print(df['ambientes'].median())

In [ ]:
#Reemplazar los NaN por 2 (valor de la mediana)
df['ambientes']=df['ambientes'].fillna(2)

### **4.2.3 Creación de variable 'ambientes_grupo'**

Se pudo observar en el análisis exploratorio que la concentración de inmuebles con siete o mas ambientes es muy baja, es por esto que se crea la variable 'ambientes_grupo' en la que en estos datos se reemplazará su valor por la mediana. Dicha variable se utilizará mas adelante.

In [ ]:
# Duplicar la columna 'ambientes' y renombrarla para agrupar los ambientes mayores o iguales a 5
df['ambientes_grupo'] = df['ambientes']

In [ ]:
filtrar_ambientes_grupo = (df['ambientes'] >= 7)
# Calcularla medina
print(df[filtrar_ambientes_grupo]['ambientes'].median())

In [ ]:
# Reemplazar los valores mayores a 7 por 7 (mediana)
df['ambientes_grupo'] = np.where(df['ambientes_grupo'] > 7, 7, df['ambientes_grupo'])

### **4.2.4 Variable 'superficie'**

In [ ]:
#Observar las filas de mayores superficies
top_mayores_sup = df.sort_values(by='superficie', ascending=False).head(int(len(df) * 0.0002))

# Visualizar las filas completas
print(top_mayores_sup)

Observando las filas, los dos de mayores superficies parecen departamentos por las características, el resto podrían ser casas. Esta aclaración es porque las casas pueden tener grandes superficies, algo poco probable en los departamentos.

In [ ]:
#Reemplazar los valores mayores a 500 por valores nulos

df['superficie'] = np.where(df['superficie'] > 500, np.nan, df['superficie'])

Por reglamentación de la ciudad de Buenos Aires, la superficie mínima de un departamento es de 18 metros cuadrados sin considerar el baño. Se utilizará esta información para la limpieza de datos, considerando que superficies menores se tratarían de errores.

In [ ]:
#Reemplazar los valores menores a 18 por valores nulos

df['superficie'] = np.where(df['superficie'] < 18, np.nan, df['superficie'])

In [ ]:
#Conteo de valores nulos
nulos_sup=df['superficie'].isnull().sum()
print(nulos_sup)

**Imputación de datos**

Se analizará la relación lineal entre las variables 'superficie' y 'ambientes' ya que debería haber relación entre ellas.

In [ ]:
#Índice de correlación de Pearson entre las variable 'superficie' y 'ambientes'
corr_coef = df['superficie'].corr(df['ambientes'])
print(corr_coef)

Se observa una fuerte relación entre las variables, para reafirmarlo se realizará un test de hipótesis.

*TEST DE HIPÓTESIS*

**Hipótesis Nula (H0):** No existe una correlación significativa entre el número de ambientes y la superficie (ρ = 0).

**Hipótesis Alternativa (H1):** Existe una correlación significativa entre el número de ambientes y la superficie (ρ ≠ 0).


In [ ]:
# Filtrar df eliminando valores nulos
df_filtrado = df.dropna(subset=['ambientes', 'superficie'])


# Calcular la correlación de Pearson y el valor p
corr, p_valor = stats.pearsonr(df_filtrado['ambientes'], df_filtrado['superficie'])

print("Coeficiente de correlación:", corr)
print("Valor p:", p_valor)

Un p-valor chico (menor a 0.05), permite descartar la hipótesis nula.

*Decisión:*
 Con una confianza del 95% se puede afirmar que existe una correlación significativa entre el número de ambientes y la superficie.


Se calculcula la recta de regresión lineal para imputar valores faltantes.

In [ ]:
# Eliminar filas con valores nulos en ambas columnas
df_filtrado_sin_nulos = df.dropna(subset=['ambientes', 'superficie'])

#Regresión lineal
x = df_filtrado_sin_nulos['ambientes'].values.reshape(-1, 1)  # Convertir a matriz
y = df_filtrado_sin_nulos['superficie']
x = sm.add_constant(x)  # Agregar la constante para la intersección
model = sm.OLS(y, x).fit()

# Obtener el resumen del modelo
print(model.summary())

La ecuación de la recta es:
superficie = -17.6055 + 34.7921 * ambientes

A continuación se imputan los valores faltantes a partir de la recta limitando los valores con el mismo criterio que se utilizó en la limpieza de datos.

In [ ]:
# Identifica los índices de las filas con valores nulos en 'superficie'
indices_nulos = df[df['superficie'].isnull()].index

# Utiliza la ecuación de la recta para predecir los valores faltantes
df.loc[indices_nulos, 'superficie'] = -17.6055 + 34.7921 * df.loc[indices_nulos, 'ambientes']

# Establece los límites
limite_inferior = 18
limite_superior = 500

# Ajusta las predicciones para que estén dentro del rango
df.loc[indices_nulos, 'superficie'] = df.loc[indices_nulos, 'superficie'].clip(lower=limite_inferior, upper=limite_superior)
df.loc[indices_nulos, 'superficie'] = -17.6055 + 34.7921 * df.loc[indices_nulos, 'ambientes']

### **4.2.5 Variable 'alq_dolar'**

Primero se realiza una limpieza de datos erróneos considerando que el alquiler en dólares no debería ser menor a 50 dólares ni mayor a 15000.

In [ ]:
#Eliminar filas en las que la variable 'alq_dolar' es menor a 50 o mayor a 10000
filtrar_alq_dolar = (df['alq_dolar'] >= 50) & (df['alq_dolar'] <= 10000)
df = df[filtrar_alq_dolar]

In [ ]:
# Agrupar los datos por 'categoria_barrio' y 'ambientes_grupo'
grupos_df = df.groupby(['categoria_barrio', 'ambientes_grupo'])


Se crean los siguientes boxplot con el objetivo de observar valores imposibles.

In [ ]:
# Crear un boxplot para cada grupo, agrupando por 'categoria_barrio' y 'ambientes_grupo'
for (barrio, ambientes), grupo in grupos_df:
    plt.figure(figsize=(15, 6))
    sns.boxplot(x='alq_dolar', data=grupo)
    plt.title(f"Distribución del precio de alquiler en la categoría {barrio} con {ambientes} ambientes")
    # Ajustar la escala del eje x de 500 en 500
    plt.xticks(np.arange(0, max(df['alq_dolar']) + 500, 500))
    plt.show()

Se observan los gráficos para eliminar valores erroneos en los grupos.


In [ ]:
# Filtrar los datos
filtro = (df['categoria_barrio'] == 1) & (df['ambientes_grupo'] == 1) & (df['alq_dolar'] > 550)
# Obtener los índices de las filas que cumplen el filtro
indices_a_eliminar = df[filtro].index
# Eliminar las filas del DataFrame
df.drop(index=indices_a_eliminar, inplace=True)

# Filtrar los datos
filtro = (df['categoria_barrio'] == 1) & (df['ambientes_grupo'] == 2) & (df['alq_dolar'] > 700)
# Obtener los índices de las filas que cumplen el filtro
indices_a_eliminar = df[filtro].index
# Eliminar las filas del DataFrame
df.drop(index=indices_a_eliminar, inplace=True)

# Filtrar los datos
filtro = (df['categoria_barrio'] == 1) & (df['ambientes_grupo'] == 3) & (df['alq_dolar'] > 950)
# Obtener los índices de las filas que cumplen el filtro
indices_a_eliminar = df[filtro].index
# Eliminar las filas del DataFrame
df.drop(index=indices_a_eliminar, inplace=True)

# Filtrar los datos
filtro = (df['categoria_barrio'] == 1) & (df['ambientes_grupo'] == 4) & (df['alq_dolar'] > 1250)
# Obtener los índices de las filas que cumplen el filtro
indices_a_eliminar = df[filtro].index
# Eliminar las filas del DataFrame
df.drop(index=indices_a_eliminar, inplace=True)

# Filtrar los datos
filtro = (df['categoria_barrio'] == 1) & (df['ambientes_grupo'] == 5) & (df['alq_dolar'] > 1300)
# Obtener los índices de las filas que cumplen el filtro
indices_a_eliminar = df[filtro].index
# Eliminar las filas del DataFrame
df.drop(index=indices_a_eliminar, inplace=True)

# Filtrar los datos
filtro = (df['categoria_barrio'] == 2) & (df['ambientes_grupo'] == 1) & (df['alq_dolar'] > 650)
# Obtener los índices de las filas que cumplen el filtro
indices_a_eliminar = df[filtro].index
# Eliminar las filas del DataFrame
df.drop(index=indices_a_eliminar, inplace=True)

# Filtrar los datos
filtro = (df['categoria_barrio'] == 2) & (df['ambientes_grupo'] == 2) & (df['alq_dolar'] > 750)
# Obtener los índices de las filas que cumplen el filtro
indices_a_eliminar = df[filtro].index
# Eliminar las filas del DataFrame
df.drop(index=indices_a_eliminar, inplace=True)

# Filtrar los datos
filtro = (df['categoria_barrio'] == 2) & (df['ambientes_grupo'] == 3) & (df['alq_dolar'] > 1000)
# Obtener los índices de las filas que cumplen el filtro
indices_a_eliminar = df[filtro].index
# Eliminar las filas del DataFrame
df.drop(index=indices_a_eliminar, inplace=True)

# Filtrar los datos
filtro = (df['categoria_barrio'] == 2) & (df['ambientes_grupo'] == 4) & (df['alq_dolar'] > 1300)
# Obtener los índices de las filas que cumplen el filtro
indices_a_eliminar = df[filtro].index
# Eliminar las filas del DataFrame
df.drop(index=indices_a_eliminar, inplace=True)

# Filtrar los datos
filtro = (df['categoria_barrio'] == 2) & (df['ambientes_grupo'] == 5) & (df['alq_dolar'] > 1450)
# Obtener los índices de las filas que cumplen el filtro
indices_a_eliminar = df[filtro].index
# Eliminar las filas del DataFrame
df.drop(index=indices_a_eliminar, inplace=True)

# Filtrar los datos
filtro = (df['categoria_barrio'] == 3) & (df['ambientes_grupo'] == 1) & (df['alq_dolar'] > 800)
# Obtener los índices de las filas que cumplen el filtro
indices_a_eliminar = df[filtro].index
# Eliminar las filas del DataFrame
df.drop(index=indices_a_eliminar, inplace=True)

# Filtrar los datos
filtro = (df['categoria_barrio'] == 3) & (df['ambientes_grupo'] == 2) & (df['alq_dolar'] > 1100)
# Obtener los índices de las filas que cumplen el filtro
indices_a_eliminar = df[filtro].index
# Eliminar las filas del DataFrame
df.drop(index=indices_a_eliminar, inplace=True)

# Filtrar los datos
filtro = (df['categoria_barrio'] == 3) & (df['ambientes_grupo'] == 3) & (df['alq_dolar'] > 1850)
# Obtener los índices de las filas que cumplen el filtro
indices_a_eliminar = df[filtro].index
# Eliminar las filas del DataFrame
df.drop(index=indices_a_eliminar, inplace=True)

# Filtrar los datos
filtro = (df['categoria_barrio'] == 3) & (df['ambientes_grupo'] == 4) & (df['alq_dolar'] > 3350)
# Obtener los índices de las filas que cumplen el filtro
indices_a_eliminar = df[filtro].index
# Eliminar las filas del DataFrame
df.drop(index=indices_a_eliminar, inplace=True)

# Filtrar los datos
filtro = (df['categoria_barrio'] == 3) & (df['ambientes_grupo'] == 5) & (df['alq_dolar'] > 6300)
# Obtener los índices de las filas que cumplen el filtro
indices_a_eliminar = df[filtro].index
# Eliminar las filas del DataFrame
df.drop(index=indices_a_eliminar, inplace=True)

### **4.2.6 Variable 'alq_peso'**

In [ ]:
# Contar los valores nulos en la columna 'alq_peso'
cantidad_nulos = print(df['alq_peso'].isnull().sum())

Esta variable depende de la variable alq_dolar, por lo que los valores imposibles probablemente se hayan limpiado ya. A continuación se verá esto en distintos boxplot.

In [ ]:
# Agrupar los datos por 'categoria_barrio' y 'ambientes_grupo'
grupos_df = df.groupby(['categoria_barrio', 'ambientes_grupo'])

# Crear un boxplot para cada grupo, agrupando por 'categoria_barrio' y 'ambientes_grupo'
for (barrio, ambientes), grupo in grupos_df:
    plt.figure(figsize=(18, 6))
    sns.boxplot(x='alq_peso', data=grupo)
    plt.title(f"Distribución del precio de alquiler en la categoría {barrio} con {ambientes} ambientes")
    # Ajustar la escala del eje x de 5000 en 5000
    (np.arange(0, max(df['alq_peso']) + 5000, 5000))
    plt.show()
    plt.show()

Si se observan los gráficos se puede ver que, si bien hay mayor variabilidad en los datos, esto es de esperarse pues el dataset tiene información durantes tres años y los alquileres en pesos variaron mucho debido a la devaluación del peso argentino. Aún así no se observan valores imposibles.

## **4.3 Escalar variables**

A continuación se escalarán las variables.

In [ ]:
# Columnas numéricas a escalar
escalar = ['alq_dolar', 'alq_peso','superficie', 'ambientes', 'ambientes_grupo', 'antiguedad', 'categoria_barrio']

In [ ]:
# Crear un objeto MinMaxScaler
scaler = MinMaxScaler()

# Ajustar el escalador y transformar
df[escalar] = scaler.fit_transform(df[escalar])

In [ ]:
#Matriz de correlación de Pearson
eliminar_columnas=['id','barrio','fecha_publ']
df_filtrado=df.drop(columns=eliminar_columnas,axis=1)
corr_matrix = df_filtrado.corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f')
plt.title('Matriz de Correlación')
plt.show()

Se puede observar en la matriz de correlación de Pearson que la relación lineal entre las variables aumentó significativamente a partir de la limpieza de datos.

La mencionada matriz se tiene encuenta para definir las variables con mayor correlación con 'alq_dolar', para lugo usarlas en la detección de outliers mediante un modelo de InsolationForest.


# **5 REDUCCIÓN DE DIMENSIONALIDAD**

Se calculan las matrices de correlación de Spearman y Kendall para observar la coorrelación entre variables con otros coeficientes y no descartar variables que puedan ser importantes.

In [ ]:
# Seleccionar las columnas de interés
columnas_seleccionadas = ['alq_dolar', 'superficie', 'ambientes','ambientes_grupo', 'antiguedad', 'gimnasio', 'lavadero',
                         'sum', 'pileta', 'balcon', 'parrilla', 'aire', 'amoblado', 'vigilancia', 'categoria_barrio']
df_filtrado = df[columnas_seleccionadas]

# Calcular las matrices de correlación
corr_spearman = df_filtrado.corr(method='spearman')
corr_kendall = df_filtrado.corr(method='kendall')

# Crear los heatmaps
plt.figure(figsize=(10, 20))

# Heatmap de Spearman
plt.subplot(3, 1, 2)
sns.heatmap(corr_spearman, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlación de Spearman')

# Heatmap de Kendall
plt.subplot(3, 1, 3)
sns.heatmap(corr_kendall, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlación de Kendall')

plt.tight_layout()
plt.show()

Se realizará una reducción de dimensionalidad de las variables secundarias: 'gimnasio', 'lavadero', 'pileta', 'balcon' y 'vigilancia', descartando aquellas variables cuya correlación con la variable target es inferior a 0.2 con todos los tres coeficientes de correlación analizados.

In [ ]:
# Filtrar df
columnas = ['gimnasio', 'lavadero', 'pileta', 'balcon', 'vigilancia']
df_filtrado = df[columnas]

# Aplicar PCA
n_components = 3
pca = PCA(n_components=n_components)
principalComponents = pca.fit_transform(df_filtrado)

# Explicar la varianza
explained_variance_ratio = pca.explained_variance_ratio_
print("Proporción de varianza explicada por cada componente:", explained_variance_ratio)


Se redujeron cinco variables a tres, con aproximadamente 0.8 de la varianza explicada como se observa en el siguiente gráfico.

In [ ]:
plt.bar(range(1, len(pca.explained_variance_ratio_) + 1), pca.explained_variance_ratio_)
plt.xlabel('Número de componente principal')
plt.ylabel('Proporción de varianza explicada')
plt.title('Varianza explicada por cada nueva componente')
plt.show()

In [ ]:
# Crear un DataFrame con los componentes principales
principalDf = pd.DataFrame(data = principalComponents,
             columns = [f'PC{i+1}' for i in range(n_components)])

In [ ]:
# Reiniciar índices antes de concatenar
df.reset_index(drop=True, inplace=True)
principalDf.reset_index(drop=True, inplace=True)

In [ ]:
# Concatenar con el DataFrame original
df = pd.concat([df, principalDf], axis=1)

In [ ]:
# Nombres de las nuevas variables
print(principalDf.columns)

# **6 MODELADO**

## *6.1 Regresión lineal múltiple*

Primero se probará si la diferencia entre el funcionamiento del modelo con las variables secundarias origiales y las que se obtuvieron a partir de la reducción de dimensionalidad es significativa.

In [ ]:
# Función para evaluar el modelo
def evaluate_model(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    return mse, rmse, r2, mae

# Seleccionar las variables independientes y dependiente
X = df[['superficie', 'antiguedad', 'amoblado', 'sum', 'aire', 'parrilla', 'ambientes', 'PC1', 'PC2', 'PC3', 'categoria_barrio']]
y = df['alq_dolar']

# Dividir los datos en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Crear el modelo de regresión lineal
model = LinearRegression()

# Ajustar el modelo
model.fit(X_train, y_train)

# Configurar la validación cruzada
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

# Realizar la validación cruzada y calcular el error (MSE)
scores = cross_val_score(model, X, y, cv=kfold, scoring='neg_mean_squared_error')

# Definir el espacio de búsqueda de hiperparámetros
param_grid = {
    'alpha': [0.01, 0.1, 1, 10, 100]  # Parámetro de regularización en Ridge
}

# Crear el modelo Ridge
ridge_model = Ridge()

# Configurar GridSearchCV
grid_search = GridSearchCV(estimator=ridge_model, param_grid=param_grid,
                           cv=kfold, scoring='neg_mean_squared_error')

# Ajustar el modelo
grid_search.fit(X, y)

# Predecir y evaluar con el mejor modelo
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X)

# Evaluar el modelo en el conjunto de entrenamiento
y_train_pred = model.predict(X_train)
train_mse, train_rmse, train_r2, train_mae = evaluate_model(y_train, y_train_pred)

# Evaluar el modelo en el conjunto de prueba
y_test_pred = model.predict(X_test)
test_mse, test_rmse, test_r2, test_mae = evaluate_model(y_test, y_test_pred)

# Imprimir los resultados
print("Mejores hiper parámetros:", grid_search.best_params_)
print("Evaluación en el conjunto de entrenamiento:")
print("MSE:", train_mse)
print("RMSE:", train_rmse)
print("R²:", train_r2)
print("MAE:", train_mae)

print("Evaluación en el conjunto de prueba:")
print("MSE:", test_mse)
print("RMSE:", test_rmse)
print("R²:", test_r2)
print("MAE:", test_mae)

# Entrenar el modelo de regresión lineal
model.fit(X, y)

# Obtener los coeficientes
coefficients = model.coef_

# Crear un DataFrame para visualizar las características y sus coeficientes
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': coefficients
})

# Ordenar por valor absoluto de los coeficientes (importancia)
feature_importance['Absolute Coefficient'] = np.abs(feature_importance['Coefficient'])
feature_importance = feature_importance.sort_values(by='Absolute Coefficient', ascending=False)

print(feature_importance)

# Configurar el estilo del gráfico
sns.set(style="whitegrid")

Se entrena el modelo fijando los hiper parámetros optimizados y eliminando las variables que aportan poco al modelo (PC1, PC2, PC3, sum, aire y parrilla)

In [ ]:
# Seleccionar las variables independientes y dependiente
X = df[['superficie','antiguedad','ambientes','amoblado','categoria_barrio']]
y = df['alq_dolar']

# Dividir los datos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Crear el modelo de regresión Ridge con alpha=0.01
model = Ridge(alpha=0.01)

# Ajustar el modelo a los datos de entrenamiento
model.fit(X_train, y_train)

# Hacer predicciones en los conjuntos de entrenamiento y prueba
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# Evaluar el modelo
def evaluate_model(y_true, y_pred):
  mse = mean_squared_error(y_true, y_pred)
  rmse = np.sqrt(mse)
  r2 = r2_score(y_true, y_pred)
  return mse, rmse, r2

# Evaluar el modelo en el conjunto de entrenamiento
train_mse, train_rmse, train_r2 = evaluate_model(y_train, y_train_pred)

# Evaluar el modelo en el conjunto de prueba
test_mse, test_rmse, test_r2 = evaluate_model(y_test, y_test_pred)

# Imprimir los resultados
print("Evaluación en el conjunto de entrenamiento:")
print("MSE:", train_mse)
print("RMSE:", train_rmse)
print("R²:", train_r2)

print("\nEvaluación en el conjunto de prueba:")
print("MSE:", test_mse)
print("RMSE:", test_rmse)
print("R²:", test_r2)

El MSE en prueba es mayor que el MSE en entrenamiento, aunque esta diferencia es mínima, lo que podría estar indicando que no hay riesgo de sobreajuste.

Si bien el modelo no produce errores altos, la predicción es moderada (observada en el coeficiente de determinación), por lo que se explorarán otros modelos para obtener mejores resultados.

## *6.2 Regresión polinómica*

Se comienza utilizando HalvinGridSearch para optimizar hiper parámetros y se realiza validación cruzada.

In [ ]:
# Seleccionar las variables independientes y dependiente
X = df[['superficie', 'antiguedad', 'sum', 'parrilla', 'aire', 'amoblado', 'PC1', 'PC2', 'PC3', 'categoria_barrio']]
y = df['alq_dolar']

# Dividir los datos en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Crear un pipeline con transformación polinómica y regresión lineal
pipeline = Pipeline([
    ('poly', PolynomialFeatures()),
    ('linear', LinearRegression())
])

# Definir el espacio de búsqueda de hiperparámetros
param_grid = {
    'poly__degree': [2, 3, 4],
    'linear__fit_intercept': [True, False]
}

# Crear el objeto de búsqueda en cuadrícula
grid_search = GridSearchCV(estimator=pipeline, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error')

# Ajustar el modelo con búsqueda en cuadrícula
grid_search.fit(X_train, y_train)

# Obtener los mejores parámetros y el mejor modelo
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_

# Obtener los mejores parámetros y el mejor modelo
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_

# Imprimir los mejores parámetros
print("Mejores parámetros:", best_params)
# Hacer predicciones con el mejor modelo
y_train_pred = best_model.predict(X_train)
y_test_pred = best_model.predict(X_test)

# Evaluar el modelo en el conjunto de entrenamiento
mse_train = mean_squared_error(y_train, y_train_pred)
rmse_train = np.sqrt(mse_train)
r2_train = r2_score(y_train, y_train_pred)

print("MSE en entrenamiento:", mse_train)
print("RMSE en entrenamiento:", rmse_train)
print("R² en entrenamiento:", r2_train)

# Evaluar el modelo en el conjunto de prueba
mse_test = mean_squared_error(y_test, y_test_pred)
rmse_test = np.sqrt(mse_test)
r2_test = r2_score(y_test, y_test_pred)

print("MSE en prueba:", mse_test)
print("RMSE en prueba:", rmse_test)
print("R² en prueba:", r2_test)

A continuación se entrena el modelo con los hiper parámetros optimizados.

In [ ]:
# Seleccionar las variables independientes y dependiente
X = df[['superficie', 'antiguedad', 'sum', 'parrilla', 'aire', 'amoblado', 'PC1', 'PC2', 'PC3', 'categoria_barrio']]
y = df['alq_dolar']

# Dividir los datos en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Crear un pipeline con transformación polinómica y regresión lineal
pipeline = Pipeline([
    ('poly', PolynomialFeatures(degree=3)),  # Fijar el grado del polinomio a 3
    ('linear', LinearRegression(fit_intercept=False))  # Fijar fit_intercept a False
])

# Ajustar el modelo
pipeline.fit(X_train, y_train)

# Hacer predicciones con el modelo ajustado
y_train_pred = pipeline.predict(X_train)
y_test_pred = pipeline.predict(X_test)

# Evaluar el modelo en el conjunto de entrenamiento
mse_train = mean_squared_error(y_train, y_train_pred)
rmse_train = np.sqrt(mse_train)
r2_train = r2_score(y_train, y_train_pred)
print("MSE en entrenamiento:", mse_train)
print("RMSE en entrenamiento:", rmse_train)
print("R² en entrenamiento:", r2_train)

# Evaluar el modelo en el conjunto de prueba
mse_test = mean_squared_error(y_test, y_test_pred)
rmse_test = np.sqrt(mse_test)
r2_test = r2_score(y_test, y_test_pred)
print("MSE en prueba:", mse_test)
print("RMSE en prueba:", rmse_test)
print("R² en prueba:", r2_test)

Se analizan las caracteríticas principales del modelo:

In [ ]:
# Seleccionar las variables independientes y dependiente
X = df[['superficie', 'antiguedad', 'sum', 'parrilla', 'aire', 'amoblado', 'PC1', 'PC2', 'PC3', 'categoria_barrio']]
y = df['alq_dolar']

# Dividir los datos en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Crear un pipeline con transformación polinómica y regresión lineal
pipeline = Pipeline([
    ('poly', PolynomialFeatures(degree=3)),  # Fijar el grado del polinomio a 3
    ('linear', LinearRegression(fit_intercept=False))  # Fijar fit_intercept a False
])

# Ajustar el modelo
pipeline.fit(X_train, y_train)

# Hacer predicciones con el modelo ajustado
y_train_pred = pipeline.predict(X_train)
y_test_pred = pipeline.predict(X_test)

# Evaluar el modelo en el conjunto de entrenamiento
mse_train = mean_squared_error(y_train, y_train_pred)
rmse_train = np.sqrt(mse_train)
r2_train = r2_score(y_train, y_train_pred)
print("MSE en entrenamiento:", mse_train)
print("RMSE en entrenamiento:", rmse_train)
print("R² en entrenamiento:", r2_train)

# Evaluar el modelo en el conjunto de prueba
mse_test = mean_squared_error(y_test, y_test_pred)
rmse_test = np.sqrt(mse_test)
r2_test = r2_score(y_test, y_test_pred)
print("MSE en prueba:", mse_test)
print("RMSE en prueba:", rmse_test)
print("R² en prueba:", r2_test)

# Calcular la importancia de las características
resultado = permutation_importance(pipeline, X_train, y_train, scoring='r2', random_state=42)

# Obtener las importancias medias
importancias = resultado.importances_mean

# Mostrar la importancia de las características
for i, importancia in enumerate(importancias):
    print(f"Importancia de {X_train.columns[i]}: {importancia:.4f}")

Se vuelve a entrenar el modelo eliminando las variables  'aire, 'sum', 'parrilla' y 'amoblado' ya que aportan poco al modelo.

In [ ]:
# Seleccionar las variables independientes y dependiente
X = df[['superficie', 'antiguedad','PC1', 'PC2', 'PC3', 'categoria_barrio']]
y = df['alq_dolar']

# Dividir los datos en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Crear un pipeline con transformación polinómica y regresión lineal
pipeline = Pipeline([
    ('poly', PolynomialFeatures(degree=3)),  # Fijar el grado del polinomio a 3
    ('linear', LinearRegression(fit_intercept=False))  # Fijar fit_intercept a False
])

# Ajustar el modelo
pipeline.fit(X_train, y_train)

# Hacer predicciones con el modelo ajustado
y_train_pred = pipeline.predict(X_train)
y_test_pred = pipeline.predict(X_test)

# Evaluar el modelo en el conjunto de entrenamiento
mse_train = mean_squared_error(y_train, y_train_pred)
rmse_train = np.sqrt(mse_train)
r2_train = r2_score(y_train, y_train_pred)
print("MSE en entrenamiento:", mse_train)
print("RMSE en entrenamiento:", rmse_train)
print("R² en entrenamiento:", r2_train)

# Evaluar el modelo en el conjunto de prueba
mse_test = mean_squared_error(y_test, y_test_pred)
rmse_test = np.sqrt(mse_test)
r2_test_poly = r2_score(y_test, y_test_pred)
print("MSE en prueba:", mse_test)
print("RMSE en prueba:", rmse_test)
print("R² en prueba:", r2_test_poly)

**1. MSE (Error Cuadrático Medio)**
- Entrenamiento: 0.0006258
- Prueba: 0.0007199

El MSE es basjo, lo que indica que el ajuste es bueno. Además se obsevan valores similares entre el MSE de entrenamiento y prueba, que se interpreta como una bajo riesgo de sobreajuste.

**2. RMSE (Raíz del Error Cuadrático Medio)**
- Entrenamiento: 0.0250
- Prueba: 0.0268

Al igual que el MSE, la diferencia entre el RMSE de entrenamiento y prueba es bajo, indicando que el modelo tiene un buen rendimiento.

**3. R² (Coeficiente de Determinación)**
- Entrenamiento: 0.7386
- Prueba: 0.7436

En este caso el R² de prueba es superior al de entrenamiento, que sin bien es poco común, es positivo que pase esto ya que se interpreta que el modelo puede estar capturando patrones para la predicción.

**Interpretación General**
- En general, los resultados muestran que el modelo tiene un buen rendimiento tanto en los datos de entrenamiento como en los de prueba.
- La pequeña diferencia entre MSE y RMSE en entrenamiento y prueba indica que el modelo no está sobreajustado y que es capaz de generalizar bien a nuevos datos.
- El R² en prueba siendo apenas superior al de entrenamiento es un buen signo, ya que sugiere que el modelo puede estar capturando patrones para la predicción.

## *6.3 Modelo SMV*

Se comienza realizando optimización de los hiper parámetros.

In [ ]:
# Seleccionar las variables independientes y dependiente
X = df[['superficie', 'antiguedad', 'sum', 'parrilla', 'aire', 'amoblado', 'PC1', 'PC2', 'PC3', 'categoria_barrio']]
y = df['alq_dolar']

# Dividir los datos en conjunto de entrenamiento y prueba (80% entrenamiento, 20% prueba)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Crear un modelo SVM con kernel RBF
svm_model = SVR(kernel='rbf')

# Definir el espacio de búsqueda de hiperparámetros
param_grid = {
    'kernel': ['rbf', 'linear', 'poly'],
    'C': [0.1, 1, 10],
    'gamma': [0.1, 0.01, 0.001]
}

# Crear el objeto de búsqueda en cuadrícula
halving_cv = HalvingGridSearchCV(estimator=svm_model, param_grid=param_grid, factor=2, resource='n_samples', max_resources=len(X_train), random_state=42)

# Ajustar el modelo con búsqueda halving
halving_cv.fit(X_train, y_train)

# Obtener los mejores parámetros y el mejor modelo
best_params = halving_cv.best_params_
best_model = halving_cv.best_estimator_

# Hacer predicciones con el mejor modelo
y_train_pred = best_model.predict(X_train)
y_test_pred = best_model.predict(X_test)

# Evaluar el modelo
mse_train = mean_squared_error(y_train, y_train_pred)
rmse_train = np.sqrt(mse_train)
r2_train = r2_score(y_train, y_train_pred)

mse_test = mean_squared_error(y_test, y_test_pred)
rmse_test = np.sqrt(mse_test)
r2_test = r2_score(y_test, y_test_pred)

print("Mejores parámetros:", best_params)
print("MSE en entrenamiento:", mse_train)
print("RMSE en entrenamiento:", rmse_train)
print("R² en entrenamiento:", r2_train)
print("MSE en prueba:", mse_test)
print("RMSE en prueba:", rmse_test)
print("R² en prueba:", r2_test)

Se entrena el modelo con los mejores hiper parámetros.

In [ ]:
# Seleccionar las variables independientes y dependiente
X = df[['superficie', 'antiguedad', 'sum', 'parrilla', 'aire', 'amoblado', 'PC1', 'PC2', 'PC3', 'categoria_barrio']]
y = df['alq_dolar']

# Dividir los datos en conjunto de entrenamiento y prueba (80% entrenamiento, 20% prueba)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Crear un modelo SVM con kernel RBF
svm_model = SVR(kernel='rbf')

# Crear un nuevo modelo SVM con los mejores parámetros
best_model = SVR(C= 10, gamma= 0.1, kernel= 'linear')

# Ajustar el modelo con búsqueda halving
halving_cv.fit(X_train, y_train)

# Obtener los mejores parámetros y el mejor modelo
best_params = halving_cv.best_params_
best_model = halving_cv.best_estimator_

# Hacer predicciones con el mejor modelo
y_train_pred = best_model.predict(X_train)
y_test_pred = best_model.predict(X_test)

# Evaluar el modelo
mse_train = mean_squared_error(y_train, y_train_pred)
rmse_train = np.sqrt(mse_train)
r2_train = r2_score(y_train, y_train_pred)

mse_test = mean_squared_error(y_test, y_test_pred)
rmse_test = np.sqrt(mse_test)
r2_test = r2_score(y_test, y_test_pred)

print("MSE en entrenamiento:", mse_train)
print("RMSE en entrenamiento:", rmse_train)
print("R² en entrenamiento:", r2_train)
print("MSE en prueba:", mse_test)
print("RMSE en prueba:", rmse_test)
print("R² en prueba:", r2_test)

De manera general se puede observar que los datos se ajustan bien, tanto en el conjunto de entrenamiento como en el de prueba. No obtante el el modelo explica alrededor del 26.8% de la variabilidad en los precios de alquiler en el conjunto de entrenamiento y 36.8% en el de prueba según el coeficiente de determinación, lo que implica que no es el mejor modelo frente a los otros.

##**6.4 Árboles de decisión**

In [ ]:
# Seleccionar las variables independientes y dependiente
X = df[['superficie', 'ambientes', 'gimnasio', 'lavadero', 'pileta', 'balcon', 'vigilancia', 'categoria_barrio']]
y = df['alq_dolar']

# Dividir los datos en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Crear un modelo de árbol de decisión
model = DecisionTreeRegressor(random_state=42)

# Definir el espacio de búsqueda de hiperparámetros
param_grid = {
    'max_depth': [None, 10, 20, 30],  # Profundidad máxima del árbol
    'min_samples_split': [2, 5, 10],  # Mínimo número de muestras para dividir un nodo
    'min_samples_leaf': [1, 2, 4],    # Mínimo número de muestras en una hoja
    'max_features': ['sqrt', 'log2', None],  # Número de características a considerar en cada división
    'criterion': ['squared_error', 'friedman_mse']  # Criterio para medir la calidad de una división
}

# Crear el objeto de búsqueda en cuadrícula con validación cruzada
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)

# Ajustar el modelo con búsqueda de hiperparámetros
grid_search.fit(X_train, y_train)

# Obtener los mejores parámetros y el mejor modelo
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_

# Hacer predicciones con el mejor modelo
y_pred = best_model.predict(X_test)

# Evaluar el modelo
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Mejores hiperparámetros:", best_params)
print("Error cuadrático medio (MSE):", mse)
print("Coeficiente de determinación (R²):", r2)

Ahora se entrena el modelo con los hiper parámetros optimizados:

In [ ]:
# Seleccionar las variables independientes y dependiente
X = df[['superficie', 'ambientes', 'gimnasio', 'lavadero', 'pileta', 'balcon', 'vigilancia', 'categoria_barrio']]
y = df['alq_dolar']

# Dividir los datos en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Crear un modelo de árbol de decisión con los hiperparámetros especificados
best_model = DecisionTreeRegressor(
    criterion='squared_error',  # Criterio para medir la calidad de una división
    max_depth=20,               # Profundidad máxima del árbol
    max_features='log2',        # Número de características a considerar en cada división
    min_samples_leaf=4,         # Mínimo número de muestras en una hoja
    min_samples_split=2,        # Mínimo número de muestras para dividir un nodo
    random_state=42             # Semilla para reproducibilidad
)

# Ajustar el modelo con los datos de entrenamiento
best_model.fit(X_train, y_train)

# Hacer predicciones en el conjunto de entrenamiento y prueba
y_train_pred = best_model.predict(X_train)  # Predicciones en entrenamiento
y_test_pred = best_model.predict(X_test)    # Predicciones en prueba

# Evaluar el modelo en el conjunto de entrenamiento
mse_train = mean_squared_error(y_train, y_train_pred)
rmse_train = np.sqrt(mse_train)  # RMSE en entrenamiento
r2_train = r2_score(y_train, y_train_pred)

# Evaluar el modelo en el conjunto de prueba
mse_test = mean_squared_error(y_test, y_test_pred)
rmse_test = np.sqrt(mse_test)  # RMSE en prueba
r2_test = r2_score(y_test, y_test_pred)

# Imprimir resultados
print("Hiperparámetros utilizados:")
print({
    'criterion': 'squared_error',
    'max_depth': 20,
    'max_features': 'log2',
    'min_samples_leaf': 4,
    'min_samples_split': 2
})
print("\nEvaluación en el conjunto de entrenamiento:")
print("Error cuadrático medio (MSE):", mse_train)
print("Raíz del error cuadrático medio (RMSE):", rmse_train)
print("Coeficiente de determinación (R²):", r2_train)

print("\nEvaluación en el conjunto de prueba:")
print("Error cuadrático medio (MSE):", mse_test)
print("Raíz del error cuadrático medio (RMSE):", rmse_test)
print("Coeficiente de determinación (R²):", r2_test)

El modelo presenta buenas métricas, por lo que se intentará mejorarlo analizando las principales características.

In [ ]:
# Seleccionar las variables independientes y dependiente
X = df[['superficie', 'ambientes', 'gimnasio', 'lavadero', 'pileta', 'balcon', 'vigilancia', 'categoria_barrio']]
y = df['alq_dolar']

# Dividir los datos en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Crear un modelo de árbol de decisión con los hiperparámetros especificados
best_model = DecisionTreeRegressor(
    criterion='squared_error',  # Criterio para medir la calidad de una división
    max_depth=20,               # Profundidad máxima del árbol
    max_features='log2',        # Número de características a considerar en cada división
    min_samples_leaf=4,         # Mínimo número de muestras en una hoja
    min_samples_split=2,        # Mínimo número de muestras para dividir un nodo
    random_state=42             # Semilla para reproducibilidad
)

# Ajustar el modelo con los datos de entrenamiento
best_model.fit(X_train, y_train)

# Hacer predicciones en el conjunto de entrenamiento y prueba
y_train_pred = best_model.predict(X_train)  # Predicciones en entrenamiento
y_test_pred = best_model.predict(X_test)    # Predicciones en prueba

# Evaluar el modelo en el conjunto de entrenamiento
mse_train = mean_squared_error(y_train, y_train_pred)
rmse_train = np.sqrt(mse_train)  # RMSE en entrenamiento
r2_train = r2_score(y_train, y_train_pred)

# Evaluar el modelo en el conjunto de prueba
mse_test = mean_squared_error(y_test, y_test_pred)
rmse_test = np.sqrt(mse_test)  # RMSE en prueba
r2_test = r2_score(y_test, y_test_pred)

# Validación cruzada para una evaluación más robusta
cv_scores = cross_val_score(best_model, X, y, cv=5, scoring='neg_mean_squared_error')
cv_rmse_scores = np.sqrt(-cv_scores)  # Convertir a RMSE

# Análisis de residuos
residuos = y_test - y_test_pred  # Residuos en el conjunto de prueba

# Importancia de características
importancias = best_model.feature_importances_
feature_importances = pd.Series(importancias, index=X.columns).sort_values(ascending=False)

# Imprimir resultados
print("Hiperparámetros utilizados:")
print({
    'criterion': 'squared_error',
    'max_depth': 20,
    'max_features': 'log2',
    'min_samples_leaf': 4,
    'min_samples_split': 2
})
print("\nEvaluación en el conjunto de entrenamiento:")
print("Error cuadrático medio (MSE):", mse_train)
print("Raíz del error cuadrático medio (RMSE):", rmse_train)
print("Coeficiente de determinación (R²):", r2_train)

print("\nEvaluación en el conjunto de prueba:")
print("Error cuadrático medio (MSE):", mse_test)
print("Raíz del error cuadrático medio (RMSE):", rmse_test)
print("Coeficiente de determinación (R²):", r2_test)

print("\nValidación cruzada (RMSE):", cv_rmse_scores)
print("RMSE promedio en validación cruzada:", cv_rmse_scores.mean())

print("\nImportancia de las características:")
print(feature_importances)

# Gráfico de importancia de características
plt.figure(figsize=(10, 6))
feature_importances.plot(kind='bar', title="Importancia de las Características")
plt.ylabel("Importancia")
plt.show()

Se simplifica el modelo eliminando las variables lavadero y balcon porque aportan muy poco.

In [ ]:
# Seleccionar las variables independientes y dependiente (sin lavadero y balcon)
X = df[['superficie', 'ambientes', 'gimnasio', 'pileta', 'vigilancia',  'categoria_barrio']]
y = df['alq_dolar']

# Dividir los datos en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Crear un modelo de árbol de decisión con los hiperparámetros fijados
best_model = DecisionTreeRegressor(
    criterion='squared_error',  # Criterio para medir la calidad de una división
    max_depth=20,               # Profundidad máxima del árbol
    max_features='log2',        # Número de características a considerar en cada división
    min_samples_leaf=4,         # Mínimo número de muestras en una hoja
    min_samples_split=2,        # Mínimo número de muestras para dividir un nodo
    random_state=42             # Semilla para reproducibilidad
)

# Ajustar el modelo con los datos de entrenamiento
best_model.fit(X_train, y_train)

# Hacer predicciones en el conjunto de entrenamiento y prueba
y_train_pred = best_model.predict(X_train)  # Predicciones en entrenamiento
y_test_pred = best_model.predict(X_test)    # Predicciones en prueba

# Evaluar el modelo en el conjunto de entrenamiento
mse_train = mean_squared_error(y_train, y_train_pred)
rmse_train = np.sqrt(mse_train)  # RMSE en entrenamiento
r2_train = r2_score(y_train, y_train_pred)

# Evaluar el modelo en el conjunto de prueba
mse_test = mean_squared_error(y_test, y_test_pred)
rmse_test = np.sqrt(mse_test)  # RMSE en prueba
r2_test_arbol = r2_score(y_test, y_test_pred)

# Validación cruzada para una evaluación más robusta
cv_scores = cross_val_score(best_model, X, y, cv=5, scoring='neg_mean_squared_error')
cv_rmse_scores = np.sqrt(-cv_scores)  # Convertir a RMSE

# Importancia de características
importancias = best_model.feature_importances_
feature_importances = pd.Series(importancias, index=X.columns).sort_values(ascending=False)

# Imprimir resultados
print("Hiperparámetros utilizados:")
print({
    'criterion': 'squared_error',
    'max_depth': 20,
    'max_features': 'log2',
    'min_samples_leaf': 4,
    'min_samples_split': 2
})
print("\nEvaluación en el conjunto de entrenamiento:")
print("Error cuadrático medio (MSE):", mse_train)
print("Raíz del error cuadrático medio (RMSE):", rmse_train)
print("Coeficiente de determinación (R²):", r2_train)

print("\nEvaluación en el conjunto de prueba:")
print("Error cuadrático medio (MSE):", mse_test)
print("Raíz del error cuadrático medio (RMSE):", rmse_test)
print("Coeficiente de determinación (R²):", r2_test)

print("\nValidación cruzada (RMSE):", cv_rmse_scores)
print("RMSE promedio en validación cruzada:", cv_rmse_scores.mean())


**1. MSE (Error Cuadrático Medio)**
- Entrenamiento: 0.000519
- Prueba: 0.000676

El MSE es bajo, lo que indica que el ajuste es bueno. Además se obsevan valores cercanos entre el MSE de entrenamiento y prueba, que se interpreta como una bajo riesgo de sobreajuste.

**2. RMSE (Raíz del Error Cuadrático Medio)**
- Entrenamiento: 0.0227
- Prueba: 0.026

Al igual que el MSE, la diferencia entre el RMSE de entrenamiento y prueba es bajo, indicando que el modelo tiene un buen rendimiento.

**3. R² (Coeficiente de Determinación)**
- Entrenamiento: 0.783
- Prueba: 0.759

El R² indica que el modelo explica aproximadamente el el 78.3% de la variabilidad de los datos de entrenamiento y el 75.9% de la variabilidad de los datos de prueba.

**Interpretación General**
- En general, los resultados muestran que el modelo tiene un buen rendimiento tanto en los datos de entrenamiento como en los de prueba.
- La pequeña diferencia entre MSE y RMSE en entrenamiento y prueba indica que el modelo no está sobreajustado y que es capaz de generalizar bien a nuevos datos.
- El R² indica que el modelo interpreta bien la variabilidad de los datos.


## **6.5 Random Forest**

In [ ]:
# Seleccionar las variables independientes y dependiente
X = df[['superficie', 'ambientes', 'gimnasio', 'lavadero', 'pileta', 'balcon', 'vigilancia', 'categoria_barrio']]
y = df['alq_dolar']

# Dividir los datos en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Crear un modelo de Random Forest
rf_model = RandomForestRegressor(random_state=42)

# Definir el espacio de búsqueda de hiperparámetros
param_grid = {
    'n_estimators': [100, 200],  # Número de árboles en el bosque
    'max_depth': [10, 20, None],  # Profundidad máxima de cada árbol
    'min_samples_split': [2, 5],  # Mínimo número de muestras para dividir un nodo
    'min_samples_leaf': [1, 2, 4],  # Mínimo número de muestras en una hoja
    'max_features': ['sqrt', 'log2']  # Número de características a considerar en cada división
}

# Crear el objeto de búsqueda en cuadrícula con validación cruzada
grid_search = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    cv=5,  # 5 folds de validación cruzada
    scoring='neg_mean_squared_error',  # Métrica a optimizar (MSE negativo)
    n_jobs=-1  # Usar todos los núcleos del procesador
)

# Ajustar el modelo con búsqueda de hiperparámetros
grid_search.fit(X_train, y_train)

# Obtener los mejores parámetros y el mejor modelo
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_

# Hacer predicciones con el mejor modelo
y_train_pred = best_model.predict(X_train)  # Predicciones en entrenamiento
y_test_pred = best_model.predict(X_test)    # Predicciones en prueba

# Evaluar el modelo en el conjunto de entrenamiento
mse_train = mean_squared_error(y_train, y_train_pred)
rmse_train = np.sqrt(mse_train)  # RMSE en entrenamiento
r2_train = r2_score(y_train, y_train_pred)

# Evaluar el modelo en el conjunto de prueba
mse_test = mean_squared_error(y_test, y_test_pred)
rmse_test = np.sqrt(mse_test)  # RMSE en prueba
r2_test = r2_score(y_test, y_test_pred)

# Validación cruzada para una evaluación más robusta
cv_scores = cross_val_score(best_model, X, y, cv=5, scoring='neg_mean_squared_error')
cv_rmse_scores = np.sqrt(-cv_scores)  # Convertir a RMSE

# Importancia de características
importancias = best_model.feature_importances_
feature_importances = pd.Series(importancias, index=X.columns).sort_values(ascending=False)

# Imprimir resultados
print("Mejores hiperparámetros encontrados:")
print(best_params)
print("\nEvaluación en el conjunto de entrenamiento:")
print("Error cuadrático medio (MSE):", mse_train)
print("Raíz del error cuadrático medio (RMSE):", rmse_train)
print("Coeficiente de determinación (R²):", r2_train)

print("\nEvaluación en el conjunto de prueba:")
print("Error cuadrático medio (MSE):", mse_test)
print("Raíz del error cuadrático medio (RMSE):", rmse_test)
print("Coeficiente de determinación (R²):", r2_test)

print("\nValidación cruzada (RMSE):", cv_rmse_scores)
print("RMSE promedio en validación cruzada:", cv_rmse_scores.mean())

print("\nImportancia de las características:")
print(feature_importances)

Se elimina la variables 'balcon' porque aporta poco al modelo y se prueba con otros hiper parámetros.

In [ ]:
# Seleccionar las variables independientes y dependiente (sin balcon)
X = df[['superficie', 'ambientes', 'gimnasio', 'pileta', 'vigilancia', 'categoria_barrio','lavadero']]
y = df['alq_dolar']

# Dividir los datos en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Crear un modelo de Random Forest
rf_model = RandomForestRegressor(random_state=42)

# Definir el espacio de búsqueda de hiperparámetros (ajustado)
param_grid = {
    'n_estimators': [100, 300, 500],  # Más árboles para mejorar el rendimiento
    'max_depth': [10, 15, 20],        # Mayor profundidad para capturar patrones complejos
    'min_samples_split': [2, 5],      # Mínimo número de muestras para dividir un nodo
    'min_samples_leaf': [1, 2],       # Mínimo número de muestras en una hoja
    'max_features': ['log2']          # Usar 'log2' como en los mejores hiperparámetros
}

# Crear el objeto de búsqueda en cuadrícula con validación cruzada
grid_search = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    cv=10,  # Más folds para una evaluación más robusta
    scoring='neg_mean_squared_error',  # Métrica a optimizar (MSE negativo)
    n_jobs=-1  # Usar todos los núcleos del procesador
)

# Ajustar el modelo con búsqueda de hiperparámetros
grid_search.fit(X_train, y_train)

# Obtener los mejores parámetros y el mejor modelo
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_

# Hacer predicciones con el mejor modelo
y_train_pred = best_model.predict(X_train)  # Predicciones en entrenamiento
y_test_pred = best_model.predict(X_test)    # Predicciones en prueba

# Evaluar el modelo en el conjunto de entrenamiento
mse_train = mean_squared_error(y_train, y_train_pred)
rmse_train = np.sqrt(mse_train)  # RMSE en entrenamiento
r2_train = r2_score(y_train, y_train_pred)

# Evaluar el modelo en el conjunto de prueba
mse_test = mean_squared_error(y_test, y_test_pred)
rmse_test = np.sqrt(mse_test)  # RMSE en prueba
r2_test = r2_score(y_test, y_test_pred)

# Validación cruzada para una evaluación más robusta
cv_scores = cross_val_score(best_model, X, y, cv=10, scoring='neg_mean_squared_error')
cv_rmse_scores = np.sqrt(-cv_scores)  # Convertir a RMSE

# Importancia de características
importancias = best_model.feature_importances_
feature_importances = pd.Series(importancias, index=X.columns).sort_values(ascending=False)

# Imprimir resultados
print("Mejores hiperparámetros encontrados:")
print(best_params)
print("\nEvaluación en el conjunto de entrenamiento:")
print("Error cuadrático medio (MSE):", mse_train)
print("Raíz del error cuadrático medio (RMSE):", rmse_train)
print("Coeficiente de determinación (R²):", r2_train)

print("\nEvaluación en el conjunto de prueba:")
print("Error cuadrático medio (MSE):", mse_test)
print("Raíz del error cuadrático medio (RMSE):", rmse_test)
print("Coeficiente de determinación (R²):", r2_test)

print("\nValidación cruzada (RMSE):", cv_rmse_scores)
print("RMSE promedio en validación cruzada:", cv_rmse_scores.mean())

print("\nImportancia de las características:")
print(feature_importances)


Se entrena el modelo con los hiper parámetros optimizados.

In [ ]:
# Seleccionar las variables independientes y dependiente
X = df[['superficie', 'ambientes', 'gimnasio', 'pileta', 'vigilancia', 'categoria_barrio','lavadero']]
y = df['alq_dolar']

# Dividir los datos en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Crear un modelo de Random Forest
rf_model = RandomForestRegressor(
    n_estimators=500,  # Número de árboles en el bosque
    max_depth=10,      # Profundidad máxima de cada árbol
    min_samples_split=2,  # Mínimo número de muestras para dividir un nodo
    min_samples_leaf=1,   # Mínimo número de muestras en una hoja
    max_features='log2',  # Número de características a considerar en cada división
    random_state=42       # Semilla para reproducibilidad
)

# Ajustar el modelo con los datos de entrenamiento
rf_model.fit(X_train, y_train)

# Hacer predicciones en el conjunto de entrenamiento y prueba
y_train_pred = rf_model.predict(X_train)  # Predicciones en entrenamiento
y_test_pred = rf_model.predict(X_test)    # Predicciones en prueba

# Evaluar el modelo en el conjunto de entrenamiento
mse_train = mean_squared_error(y_train, y_train_pred)
rmse_train = np.sqrt(mse_train)  # RMSE en entrenamiento
r2_train = r2_score(y_train, y_train_pred)

# Evaluar el modelo en el conjunto de prueba
mse_test = mean_squared_error(y_test, y_test_pred)
rmse_test = np.sqrt(mse_test)  # RMSE en prueba
r2_test_RF = r2_score(y_test, y_test_pred)

# Validación cruzada para una evaluación más robusta
cv_scores = cross_val_score(rf_model, X, y, cv=5, scoring='neg_mean_squared_error')
cv_rmse_scores = np.sqrt(-cv_scores)  # Convertir a RMSE

# Imprimir resultados
print("Hiperparámetros utilizados:")
print({
    'n_estimators': 500,
    'max_depth': 10,
    'min_samples_split': 2,
    'min_samples_leaf': 1,
    'max_features': 'log2',
    'random_state': 42
})
print("\nEvaluación en el conjunto de entrenamiento:")
print("Error cuadrático medio (MSE):", mse_train)
print("Raíz del error cuadrático medio (RMSE):", rmse_train)
print("Coeficiente de determinación (R²):", r2_train)

print("\nEvaluación en el conjunto de prueba:")
print("Error cuadrático medio (MSE):", mse_test)
print("Raíz del error cuadrático medio (RMSE):", rmse_test)
print("Coeficiente de determinación (R²):", r2_test_RF)

print("\nValidación cruzada (RMSE):", cv_rmse_scores)
print("RMSE promedio en validación cruzada:", cv_rmse_scores.mean())

**1. Error Cuadrático Medio (MSE)**

- Entrenamiento: 0.000488
- Prueba: 0.000643
El MSE es bajo tanto en el conjunto de entrenamiento como en el de prueba, lo que indica un buen ajuste del modelo a los datos. Además, la diferencia entre ambos valores es relativamente pequeña, sugiriendo un bajo riesgo de sobreajuste.

**2. Raíz del Error Cuadrático Medio (RMSE)**

- Entrenamiento: 0.02209
- Prueba: 0.02536
Al igual que el MSE, el RMSE es bajo en ambos conjuntos, lo que confirma el buen desempeño del modelo. La diferencia entre los valores de entrenamiento y prueba es pequeña, indicando que el modelo generaliza bien a nuevos datos.

**3. Coeficiente de Determinación (R²)**

- Entrenamiento: 0.7962
- Prueba: 0.7709
El R² indica que el modelo explica aproximadamente el 79.6% de la variabilidad de los datos de entrenamiento y el 77.1% de la variabilidad de los datos de prueba. Estos valores son relativamente altos, lo que sugiere que el modelo captura una gran parte de la variabilidad en los datos.

**Interpretación General**

En general, los resultados muestran que el modelo tiene un buen rendimiento tanto en los datos de entrenamiento como en los de prueba. La pequeña diferencia entre los valores de MSE y RMSE en entrenamiento y prueba indica que el modelo no está sobreajustado y que es capaz de generalizar bien a nuevos datos. El R² elevado indica que el modelo explica una gran proporción de la variabilidad en los datos.

## **6.6 XGBoost**

XGBoost (eXtreme Gradient Boosting) es un algoritmo de gradient boosting optimizado que en general supera a Random Forest en precisión sobre datos tabulares, con tiempos de entrenamiento competitivos. Ya fue importado en la sección de librerías (`import xgboost as xgb`). Se usa `HalvingGridSearchCV` para reducir el tiempo de búsqueda de hiperparámetros.

In [ ]:
# Seleccionar variables (mismas que Random Forest)
X = df[['superficie', 'ambientes', 'gimnasio', 'pileta', 'vigilancia', 'categoria_barrio', 'lavadero']]
y = df['alq_dolar']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Espacio de búsqueda de hiperparámetros
param_grid_xgb = {
    'n_estimators':     [100, 300, 500],
    'max_depth':        [3, 5, 7],
    'learning_rate':    [0.01, 0.05, 0.1],
    'subsample':        [0.7, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.9, 1.0]
}

xgb_base = xgb.XGBRegressor(objective='reg:squarederror', random_state=42, verbosity=0)

halving_xgb = HalvingGridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid_xgb,
    factor=3,
    resource='n_samples',
    max_resources=len(X_train),
    scoring='neg_mean_squared_error',
    cv=5,
    n_jobs=-1,
    random_state=42
)

halving_xgb.fit(X_train, y_train)
print('Mejores hiperparámetros XGBoost:', halving_xgb.best_params_)

Se entrena el modelo XGBoost final con los hiperparámetros óptimos y se evalúan sus métricas.

In [ ]:
best_xgb = halving_xgb.best_estimator_

y_train_pred_xgb = best_xgb.predict(X_train)
y_test_pred_xgb  = best_xgb.predict(X_test)

mse_train_xgb  = mean_squared_error(y_train, y_train_pred_xgb)
rmse_train_xgb = np.sqrt(mse_train_xgb)
r2_train_xgb   = r2_score(y_train, y_train_pred_xgb)

mse_test_xgb   = mean_squared_error(y_test, y_test_pred_xgb)
rmse_test_xgb  = np.sqrt(mse_test_xgb)
r2_test_xgb    = r2_score(y_test, y_test_pred_xgb)

cv_xgb = cross_val_score(best_xgb, X, y, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
cv_rmse_xgb = np.sqrt(-cv_xgb)

print('Evaluación en entrenamiento:')
print(f'  MSE:  {mse_train_xgb:.6f}')
print(f'  RMSE: {rmse_train_xgb:.6f}')
print(f'  R²:   {r2_train_xgb:.4f}')
print('Evaluación en prueba:')
print(f'  MSE:  {mse_test_xgb:.6f}')
print(f'  RMSE: {rmse_test_xgb:.6f}')
print(f'  R²:   {r2_test_xgb:.4f}')
print(f'RMSE promedio validación cruzada: {cv_rmse_xgb.mean():.6f}')

In [ ]:
# Importancia de características — XGBoost
xgb_imp = pd.Series(best_xgb.feature_importances_, index=X.columns).sort_values(ascending=False)
plt.figure(figsize=(9, 4))
xgb_imp.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Importancia de características - XGBoost')
plt.ylabel('Importancia (F-score)')
plt.tight_layout()
plt.show()
print(xgb_imp)

**Ventajas de XGBoost sobre Random Forest:**

- **Regularización integrada** (L1/L2) → menor sobreajuste.
- **Gradient boosting secuencial** → aprende de los errores de árboles anteriores.
- **Velocidad** → implementación optimizada en C++ con paralelismo.
- **Manejo nativo de valores faltantes** en el árbol (aunque el dataset ya fue imputado).

# 7 SELECCIÓN DEL MODELO

En este proyecto se trabajó con cinco modelos diferentes: regresión lineal múltiple, regresión polinómica de tercer grado, SMV, árboles de decisión y random forest.
Todos los modelos presentaron buenas métricas en los conjuntos de entrenamiento y prueba tanto en el error cuadrático medio como en la raíz del error cuadrático medio, lo que indicaría que que el ajuste es bueno. Además se obsevan valores cercanos entre estas dos métricas en entrenamiento y prueba, que se interpreta como una bajo riesgo de sobreajuste.
Por el contrario, los coeficientes de determinación fueron muy diferentes: en la regresión lineal múltiple apenas superó el 60% y en el modelo SMV se encontró por encima del 30%, por lo que fueron los primeros dos modelos descartados. Mientras tanto, los modelos de regresión polinómica, árboles de decisión  y random forest fueron los que mejor capturaron la variabilidad de los datos por lo que sla elección del modelo se encuentra entre estos tres.. A continuación se pueden observar sus R².


In [ ]:
# ── Comparación de todos los modelos ──────────────────────
# Recalcular Regresión Lineal Ridge para incluir en la tabla
X_lr = df[['superficie','antiguedad','ambientes','amoblado','categoria_barrio']]
y_lr = df['alq_dolar']
X_tr_lr, X_te_lr, y_tr_lr, y_te_lr = train_test_split(X_lr, y_lr, test_size=0.2, random_state=42)
ridge_final = Ridge(alpha=0.01).fit(X_tr_lr, y_tr_lr)
r2_test_lr  = r2_score(y_te_lr, ridge_final.predict(X_te_lr))

resumen = pd.DataFrame({
    'Modelo': [
        'Regresión Lineal (Ridge)',
        'Regresión Polinómica (grado 3)',
        'Árbol de Decisión',
        'Random Forest',
        'XGBoost'
    ],
    'R² (prueba)': [
        round(r2_test_lr,    4),
        round(r2_test_poly,  4),
        round(r2_test_arbol, 4),
        round(r2_test_RF,   4),
        round(r2_test_xgb,  4)
    ]
}).sort_values('R² (prueba)', ascending=False).reset_index(drop=True)

print('=== Comparación de modelos (R² en conjunto de prueba) ===')
print(resumen.to_string(index=False))

# Gráfico comparativo
colors = ['#4CAF50','#2196F3','#FF9800','#9C27B0','#F44336']
plt.figure(figsize=(10, 5))
bars = plt.bar(resumen['Modelo'], resumen['R² (prueba)'],
               color=colors[:len(resumen)], edgecolor='white')
plt.title('Comparación de modelos - R² en conjunto de prueba', fontsize=13)
plt.ylabel('R²')
plt.xticks(rotation=20, ha='right')
plt.ylim(0, 1.05)
for bar, val in zip(bars, resumen['R² (prueba)']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()
print(f"\n✅ Mejor modelo: {resumen.iloc[0]['Modelo']} (R² = {resumen.iloc[0]['R² (prueba)']:.4f})")

Entre estos tres modelos, random forest presentó un rendimiento ligeramente superior, además es mas robusto y menos propenso a sobreajuste que los anteriores, por lo que se lo considera el mejor modelo para la predicción de alquileres.

# **9 FUNCIÓN DE PREDICCIÓN**

Utilidad reutilizable para estimar el precio de alquiler en USD para un nuevo departamento, usando el mejor modelo (XGBoost). Desnormaliza la predicción usando los parámetros del `MinMaxScaler` ajustado durante el preprocesamiento.

In [ ]:
def predecir_alquiler(modelo, scaler_obj, superficie, ambientes,
                       gimnasio=0, pileta=0, vigilancia=0,
                       categoria_barrio=1, lavadero=0):
    """
    Estima el precio de alquiler en USD para un departamento.

    Parámetros
    ----------
    modelo           : modelo entrenado compatible con .predict()
    scaler_obj       : MinMaxScaler ajustado sobre el dataframe
    superficie       : m² cubiertos  (ej. 45)
    ambientes        : cantidad de ambientes  (ej. 2)
    gimnasio         : 0 o 1
    pileta           : 0 o 1
    vigilancia       : 0 o 1
    categoria_barrio : 0=bajo, 1=medio, 2=alto
    lavadero         : 0 o 1

    Retorna
    -------
    precio_usd : estimación del alquiler en USD
    """
    features = ['superficie','ambientes','gimnasio','pileta','vigilancia','categoria_barrio','lavadero']

    # El scaler fue ajustado sobre:
    # ['alq_dolar','alq_peso','superficie','ambientes','ambientes_grupo','antiguedad','categoria_barrio']
    col_scaler_idx = {'superficie': 2, 'ambientes': 3, 'categoria_barrio': 6}

    entrada = {'superficie': superficie, 'ambientes': ambientes,
               'gimnasio': gimnasio, 'pileta': pileta,
               'vigilancia': vigilancia, 'categoria_barrio': categoria_barrio,
               'lavadero': lavadero}

    row = pd.DataFrame([entrada])

    # Normalizar características continuas con los parámetros del scaler
    for col, idx in col_scaler_idx.items():
        mn = scaler_obj.data_min_[idx]
        mx = scaler_obj.data_max_[idx]
        row[col] = (row[col] - mn) / (mx - mn)

    pred_norm = modelo.predict(row[features])[0]

    # Desnormalizar (alq_dolar es índice 0 en el scaler)
    mn_d = scaler_obj.data_min_[0]
    mx_d = scaler_obj.data_max_[0]
    precio_usd = pred_norm * (mx_d - mn_d) + mn_d
    return round(precio_usd, 2)


# ── Ejemplos de uso ──────────────────────────────────────────
ejemplos = [
    {'superficie':40, 'ambientes':2, 'categoria_barrio':1, 'gimnasio':0, 'pileta':0, 'vigilancia':0, 'lavadero':0},
    {'superficie':70, 'ambientes':3, 'categoria_barrio':2, 'gimnasio':1, 'pileta':1, 'vigilancia':1, 'lavadero':1},
    {'superficie':25, 'ambientes':1, 'categoria_barrio':0, 'gimnasio':0, 'pileta':0, 'vigilancia':0, 'lavadero':0},
]

print(f'{"Sup":>5} {"Amb":>5} {"Cat":>5} {"Gym":>5} {"Pileta":>7} {"Vigil":>7} {"USD estimado":>14}')
print('-' * 60)
for ej in ejemplos:
    precio = predecir_alquiler(best_xgb, scaler, **ej)
    print(f"{ej['superficie']:>5}m² {ej['ambientes']:>4}amb  cat:{ej['categoria_barrio']}  "
          f"{ej['gimnasio']:>4}  {ej['pileta']:>6}  {ej['vigilancia']:>6}  ${precio:>12,.2f}")

# 8 CONCLUSIONES

En este proyecto se desarrolló un modelo de predicción de precios de alquiler en la Ciudad Autónoma de Buenos Aires con el objetivo de brindar una herramienta útil a propietarios e inquilinos. Se utilizó un conjunto de datos de alquileres recopilado de fuentes oficiales del gobierno de la ciudad y se exploraron diversos modelos de regresión

 Los resultados mostraron que los modelosde regresión polinómica de tercer grado, árboles de decisión y random forest obtuvieron el mejor rendimiento, logrando explicar aproximadamente entre el 74% y el 80% de la variabilidad en los precios de alquiler, siendo este útimo el que alcanzó un rendimiento superior. Los modelos identificaron como variables más influyentes en el precio: la superficie, la cantidad de ambientes y el barrio donde se ubica, aunque en algunos tambien influyeron otras variables como la antigüedad del inmueble y la presencia de pileta o gimnasio.


Es importante destacar que, si bien los modelos desarrollados presentan un buen desempeño, existen factores externos que pueden influir en los precios de alquiler y que no están completamente capturados por los modelos. Como por ejemplo, las decisiones individuales de los propietarios,  factores estacionales, ciclos económicos y cambios en las políticas gubernamentales pueden afectar los precios. Además, el modelo se entrenó con datos entre los años 2017 y 2020, por lo que su desempeño podría verse afectado por cambios en el mercado inmobiliario ocurridos posteriormente. Por estos motivos, si bien el modelo presentó un buen rendimiento, no se pudo probar la hipótesis principal que afirmaba que el precio de alquiler en dólares para departamentos con características similares se mantiene relativamente estable en el tiempo, ajustando por la inflación en dólares. Por otra parte, las hipótesis secundarias panteaban que el precio de alquiler en dólares aumenta proporcionalmente con la superficie cubierta y la cantidad de ambientes y que
hay mayor oferta de inmuebles de pocos ambientes y superficies relativamente chicas". Con respecto a estas, se pudo observar que la relación lineal positiva entre el precio del alquiler en dólares y la superficie en relativamente alta (superando el 80%), mientras la relación con la cantidad de ambientes fue moderada pero también significativa. También se observó que 75% de los inmuebles tiene una superficie menor a 75 m²  y menos de 4 ambientes y un 50% tiene una superficie menor a 50 m²  y uno o dos ambientes, pudiendo validar lo panteado como hipótesis alternativa. Otro resultado significativo es la alta variabilidad en los precios de alquileres con la misma cantidad de ambientes ubicados en distintos barrios, sobre todo en aquellos que se encuentran en los barrios mas caros, tienen valores muy superiores al resto.


Para mejorar el rendimiento de los modelos se podría considerar en futuras investigaciones:

Incorporar datos más recientes: Actualizar el conjunto de datos con información más reciente permitiría mejorar la precisión de las predicciones y capturar tendencias más actuales del mercado.

Análisis de sensibilidad: Realizar un análisis de sensibilidad para evaluar cómo los cambios en los valores de las variables independientes afectan las predicciones del modelo.

Incorporar alguna variable capaz de capturar los factores inflacionarios del mercado.

En resúmen, los resultados de este proyecto demuestran que es posible desarrollar modelos de predicción de precios de alquiler con un buen nivel de precisión. Sin embargo, es importante reconocer las limitaciones de los modelos y considerar la actualización periódica de los datos para mejorar su desempeño. Este tipo de herramientas pueden ser de gran utilidad para propietarios, inquilinos y empresas del sector inmobiliario al proporcionar información valiosa para la toma de decisiones.